# CKD Guideline Retrieval Pipeline — v2

Research prototype on synthetic data. **Not** a clinical decision-support system.

This notebook focuses only on **retrieval**. The generation / recommendation step from
`initial_experiment.ipynb` is plugged in at the end once retrieval quality is measured.

## Pipeline

```text
                     OFFLINE (once)                                   ONLINE (per visit)
KDIGO PDF ─► docling ─► clean glyphs ─► statement units         clinician note + EHR history
                                     (rec / practice point           │
                                      + heading path                 ▼
                                      + section rationale      PatientState (LLM extract + merge)
                                      + linked tables)               │ deterministic derivations
                                            │                        │ (G/A category, adult, drug classes,
                                            ▼                        │  missing data)
                              applicability metadata                 ▼
                              (regex → optional LLM → human   1. eligibility filter (exclusion only)
                               review CSV)                    2. care-domain sub-queries
                                            │                 3. BM25 + dense  → RRF fusion
                                            └──────────────►  4. cross-encoder rerank
                                                              5. evidence package (+ "needs ACR" flags)
```

## Why the first attempt failed (verified)

1. Regex `\d+\.\d+\.\d+\.\d+` only matched 4-level IDs → **82 of 169 statements were never indexed**
   (e.g. Rec 3.4.1 BP target, 3.6.x RASi, 3.7.x SGLT2i, all of Chapter 2).
2. PDF glyphs: `≥` is extracted as `‡`, `≤` as `£`, `=` as `¼`, `+` as `þ` → thresholds like
   "aged ‡50" are unreadable for regex and confusing for an LLM.
3. Rationale text, subheadings (e.g. *"Pediatric considerations."*) and tables were dropped,
   so population context was lost.
4. Every statement appears twice (summary section p.34–53 and chapter body p.54+).

## 0. Setup

In [1]:
import hashlib
import json
import math
import re
import threading
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", 120)

GUIDELINE_PATH = Path("data/guidelines/kdigo_2024_ckd.pdf")
DOCLING_CACHE = Path("data/cache/kdigo_2024_docling.json")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# LLM steps cost money / send data to OpenAI. Everything else runs locally.
RUN_LLM = True
LLM_MODEL = "gpt-4.1"

# Embedding backend: "local" (sentence-transformers) or "openai"
EMBEDDING_BACKEND = "local"
LOCAL_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

# Cross-encoder rerankers to compare
RERANKERS = {
    "bge-reranker-v2-m3": "BAAI/bge-reranker-v2-m3",
    "medcpt": "ncbi/MedCPT-Cross-Encoder",
}

# Every LLM answer is cached on disk, so re-running the notebook costs nothing
# (delete the file to force fresh calls, e.g. after changing a prompt or model).
LLM_CACHE_PATH = Path(f"data/cache/llm_cache_{LLM_MODEL}.json")
_llm_cache = json.loads(LLM_CACHE_PATH.read_text()) if LLM_CACHE_PATH.exists() else {}
_llm_cache_lock = threading.Lock()


def llm_cached(key, fn):
    if key not in _llm_cache:
        value = fn()
        with _llm_cache_lock:
            _llm_cache[key] = value
            LLM_CACHE_PATH.write_text(json.dumps(_llm_cache, indent=1, ensure_ascii=False))
    return _llm_cache[key]


if RUN_LLM:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))


## 1. Parse (cached)

Docling is run once with OCR off (the PDF is born-digital) and cached as JSON.
Re-parsing takes ~45 s; loading the cache takes ~2 s.

In [2]:
from docling_core.types.doc import DoclingDocument, SectionHeaderItem, TableItem, PictureItem, TextItem, ListItem

if DOCLING_CACHE.exists():
    doc = DoclingDocument.load_from_json(DOCLING_CACHE)
else:
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    opts = PdfPipelineOptions(do_ocr=False, do_table_structure=True)
    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
    )
    doc = converter.convert(GUIDELINE_PATH).document
    DOCLING_CACHE.parent.mkdir(parents=True, exist_ok=True)
    doc.save_as_json(DOCLING_CACHE)

# Reading-order traversal, body only (drops running headers/footers = "furniture")
items = [
    item
    for item, _level in doc.iterate_items()
    if str(getattr(item, "content_layer", "")) == "ContentLayer.BODY"
]
print(len(items), Counter(type(i).__name__ for i in items))

3421 Counter({'ListItem': 1460, 'TextItem': 1443, 'SectionHeaderItem': 342, 'PictureItem': 98, 'TableItem': 78})


## 2. Clean PDF glyphs

The KDIGO PDF uses a symbol font that pypdf/docling decode incorrectly. Verified mappings
(by inspecting context, e.g. *"aged ‡50 years"*, *"n ¼ 720,736"*, *"aged 75þ"*):

In [3]:
GLYPH_MAP = {
    "‡": "≥",
    "£": "≤",
    "¼": "=",
    "þ": "+",
    "ª": "©",
    "/C15": "•",
    "$": "≥",   # in tables; all 47 occurrences are "≥" (e.g. "eGFR $60", "ACR $30 mg/g")
    "–": "-",   # en dash: docling and pypdf disagree, normalise both
    "ﬁ": "fi",
    "ﬂ": "fl",
    "­": "",  # soft hyphen
}


def map_glyphs(text: str) -> str:
    for bad, good in GLYPH_MAP.items():
        text = text.replace(bad, good)
    # re-join words hyphenated across line breaks: "recom- mend" -> "recommend"
    text = re.sub(r"(\w)-[ \n](?=[a-z])", r"\1", text)
    return re.sub(r"[ \t]+", " ", text)  # keep newlines (tables, list items)

### Repair missing spaces

Docling drops spaces inside some list items (`eGFR≥20ml/minper1.73m 2 withurineACR≥200`),
which breaks both BM25 and threshold regexes. pypdf keeps those spaces, so for any
suspiciously long unspaced token we locate the same characters in the pypdf text
(ignoring whitespace) and copy pypdf's spacing back.

In [4]:
from pypdf import PdfReader

PYPDF_CACHE = Path("data/cache/kdigo_2024_pypdf.txt")
if not PYPDF_CACHE.exists():
    PYPDF_CACHE.write_text(
        "\n".join((p.extract_text() or "") for p in PdfReader(str(GUIDELINE_PATH)).pages),
        encoding="utf-8",
    )

_ref_text = map_glyphs(PYPDF_CACHE.read_text(encoding="utf-8"))
_ref_positions = [i for i, c in enumerate(_ref_text) if not c.isspace()]
_ref_despaced = "".join(_ref_text[i] for i in _ref_positions)


def respace(text: str) -> str:
    def fix(match):
        token = match.group(0)
        pos = _ref_despaced.find(token)
        if pos < 0:
            return token
        spaced = _ref_text[_ref_positions[pos] : _ref_positions[pos + len(token) - 1] + 1]
        return re.sub(r"\s+", " ", spaced)

    # 12+ non-space chars containing a lowercase letter run glued to another word
    return re.sub(r"\S{12,}", fix, text)


def clean_text(text: str) -> str:
    text = respace(map_glyphs(text))
    text = re.sub(r"1\.73 m\s*2\b", "1.73 m2", text)  # "1.73 m 2" -> "1.73 m2"
    return text.strip()


print(clean_text("In adults aged ‡50 years with eGFR <60 ml/min per 1.73 m 2 we recom- mend a statin"))
print(clean_text("eGFR‡20ml/minper1.73m 2 withurineACR‡200 mg/g (‡20 mg/mmol), or"))
print(clean_text("(i) reviewofpastmeasurements/estimationsofGFR;"))

In adults aged ≥50 years with eGFR <60 ml/min per 1.73 m2 we recommend a statin
eGFR ≥20 ml/min per 1.73 m2 with urine ACR ≥200 mg/g (≥20 mg/mmol), or
(i) review of past measurements/estimations of GFR;


## 3. Statement units with heading path + section rationale

Walk the body in reading order and keep a heading stack:

- `Chapter N: ...` → chapter
- `3.15.1 Lipid management` → numbered section (depth from the number)
- other headers (`Special considerations`, `Pediatric considerations.`) → **subheading**,
  reset when a new numbered section starts. This is where population context often lives.

A statement = `Recommendation|Practice Point` line + following `ListItem`s.
All other prose in the same leaf section is kept as **section rationale** (parent context
for the LLM, not embedded).

In [5]:
STATEMENT_RE = re.compile(r"^(Recommendation|Practice Point)\s+(\d+(?:\.\d+){1,4})\s*:\s*")
NUMBERED_HEADER_RE = re.compile(r"^(\d+(?:\.\d+)+)\s+\S")
CHAPTER_RE = re.compile(r"^Chapter\s+(\d+)\s*[:.]\s*(.+)")
GRADE_RE = re.compile(r"\((\d[A-D])\)\s*\.?\s*$")
TABLE_REF_RE = re.compile(r"\b(Table|Figure)\s+(\d+)\b")


def find_header_index(pattern, start=0):
    for i in range(start, len(items)):
        it = items[i]
        if isinstance(it, SectionHeaderItem) and re.match(pattern, it.text.strip()):
            return i
    raise ValueError(pattern)


SUMMARY_START = find_header_index(r"Summary of recommendation statements")
BODY_START = find_header_index(r"Chapter 1:", SUMMARY_START + 2)
BODY_END = find_header_index(r"Chapter 6:", BODY_START)  # research recommendations — excluded
print("summary items", SUMMARY_START, "->", BODY_START, "| body items", BODY_START, "->", BODY_END)

summary items 392 -> 761 | body items 761 -> 1985


In [6]:
def page_of(item):
    return item.prov[0].page_no if item.prov else None


section_titles = {}  # "3.4" -> "3.4 Blood pressure control"


def walk_statements(start, end, source):
    statements = []
    section_prose = defaultdict(list)   # leaf section id -> rationale paragraphs
    chapter = section = None
    section_title = None
    subheading = None
    current = None

    for item in items[start:end]:
        if isinstance(item, (TableItem, PictureItem)):
            continue
        text = clean_text(getattr(item, "text", "") or "")
        if not text:
            continue

        # Checked before the item type: docling sometimes labels a statement as a
        # ListItem or SectionHeaderItem (e.g. PP 2.2.5, PP 3.3.1.3)
        m = STATEMENT_RE.match(text)
        if m:
            kind = "recommendation" if m.group(1) == "Recommendation" else "practice_point"
            grade = GRADE_RE.search(text)
            current = {
                "statement_id": f"{'Rec' if kind == 'recommendation' else 'PP'} {m.group(2)}",
                "number": m.group(2),
                "type": kind,
                "grade": grade.group(1) if grade else None,
                "chapter": chapter,
                "section": section,
                "section_title": section_title,
                "subheading": subheading,
                "text": text,
                "page": page_of(item),
                "source": source,
            }
            statements.append(current)
            continue

        if isinstance(item, SectionHeaderItem):
            current = None
            m_ch = CHAPTER_RE.match(text)
            m_num = NUMBERED_HEADER_RE.match(text)
            if m_ch:
                chapter, section, section_title, subheading = f"Chapter {m_ch.group(1)}: {m_ch.group(2)}", None, None, None
            elif m_num:
                section, section_title, subheading = m_num.group(1), text, None
                section_titles[m_num.group(1)] = text
            elif not re.match(r"^(Table|Figure)\s+\d+", text):
                subheading = text.rstrip(".")
            continue

        if isinstance(item, ListItem) and current is not None:
            current["text"] += "\n" + text
            continue

        # any other prose: rationale for the leaf section; ends list attachment
        current = None
        if section:
            section_prose[section].append(text)

    return statements, section_prose


summary_statements, _ = walk_statements(SUMMARY_START, BODY_START, "summary")
body_statements, section_prose = walk_statements(BODY_START, BODY_END, "body")

print("summary:", len(summary_statements), "body:", len(body_statements))
print("unique summary:", len({s['statement_id'] for s in summary_statements}),
      "unique body:", len({s['statement_id'] for s in body_statements}))

summary: 162 body: 168
unique summary: 162 unique body: 168


### Deduplicate: prefer the body copy

The body copy carries subheadings and rationale. If the body copy of a statement is
truncated by a page break, fall back to the longer summary text.

In [7]:
def dedupe(summary, body):
    by_id = {}
    for s in body:
        # a statement can legitimately be repeated inside the body (e.g. restated in a figure);
        # keep the first body occurrence
        by_id.setdefault(s["statement_id"], dict(s))
    summary_by_id = {}
    for s in summary:
        summary_by_id.setdefault(s["statement_id"], s)

    for sid, s in summary_by_id.items():
        if sid not in by_id:
            by_id[sid] = dict(s)
        elif len(s["text"]) > len(by_id[sid]["text"]) + 20:
            by_id[sid]["text"] = s["text"]
            by_id[sid]["text_from_summary"] = True

    def sort_key(sid):
        kind, num = sid.split(" ")
        return [int(x) for x in num.split(".")] + [0 if kind == "Rec" else 1]

    return [by_id[k] for k in sorted(by_id, key=sort_key)]


statements = dedupe(summary_statements, body_statements)

for s in statements:
    s["section_rationale"] = "\n".join(section_prose.get(s["section"], []))
    s["heading_path"] = " › ".join(x for x in [s["chapter"], s["section_title"], s["subheading"]] if x)
    s["table_refs"] = sorted({f"{k} {n}" for k, n in TABLE_REF_RE.findall(s["text"])})
    # care domain = the 2-level section the statement belongs to, e.g. "3.4 Blood pressure control"
    s["domain"] = section_titles.get(".".join(s["number"].split(".")[:2]), s["chapter"])

stmt_df = pd.DataFrame(statements)
print(len(statements), "unique statements")
print(stmt_df["type"].value_counts().to_dict())
print(stmt_df.groupby(stmt_df["number"].str.split(".").str[0]).size().to_dict())

169 unique statements
{'practice_point': 141, 'recommendation': 28}
{'1': 38, '2': 13, '3': 77, '4': 18, '5': 23}


In [8]:
# Completeness check against an independent extraction (pypdf, whole document, chapters 1-5)
pypdf_ids = {
    f"{'Rec' if kind == 'Recommendation' else 'PP'} {num}"
    for kind, num in re.findall(r"(Recommendation|Practice Point)\s+(\d+(?:\.\d+){1,4})\s*:", _ref_text)
    if num.split(".")[0] in {"1", "2", "3", "4", "5"}
}
ours = {s["statement_id"] for s in statements}
print("pypdf:", len(pypdf_ids), "| ours:", len(ours))
print("missing:", sorted(pypdf_ids - ours), "| extra:", sorted(ours - pypdf_ids))
assert pypdf_ids == ours

pypdf: 169 | ours: 169
missing: [] | extra: []


In [9]:
stmt_df = pd.DataFrame(statements)
stmt_df.groupby("domain").size().sort_index()

domain
1.1 Detection and evaluation of CKD                                                        9
1.2 Evaluation of GFR                                                                     19
1.3 Evaluation of albuminuria                                                              6
1.4 Point-of-care testing                                                                  4
2.1 Overview on monitoring for progression of CKD based upon GFR and ACR categories        5
2.2 Risk prediction in people with CKD                                                     6
2.3 Prediction of cardiovascular risk in people with CKD                                   2
3.1 CKD treatment and risk modi fication                                                   1
3.10 Metabolic acidosis                                                                    2
3.11 Hyperkalemia in CKD                                                                   4
3.14 Hyperuricemia                                             

Every statement is now one self-contained unit. Example of what gets indexed vs what is
kept as parent context:

In [10]:
s = next(x for x in statements if x["statement_id"] == "Rec 3.7.2")
print("INDEXED TEXT:\n", s["heading_path"], "\n", s["text"])
print("\nPARENT CONTEXT (rationale, first 600 chars):\n", s["section_rationale"][:600])

INDEXED TEXT:
 Chapter 3: Delaying CKD progression and managing its complications › 3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i) 
 Recommendation 3.7.2: We recommend treating adults with CKD with an SGLT2i for the following (1A):
 eGFR ≥20 ml/min per 1.73 m2 with urine ACR ≥200 mg/g (≥20 mg/mmol), or
 heart failure, irrespective of level of albuminuria.

PARENT CONTEXT (rationale, first 600 chars):
 The Work Group concurs with the KDIGO 2022 Clinical Practice Guideline for Diabetes Management in Chronic Kidney Disease, which stated: 'We recommend treating patients with type 2 diabetes (T2D), CKD, and an eGFR ≥20 ml/ min per 1.73 m2 with an SGLT2i (1A).' 23 However, in this guideline, we offer a more general 1A recommendation for adults with CKD. We also highlight practice points from the KDIGO Diabetes guideline for diabetes management in CKD, which are also relevant for people with CKD without diabetes:
Use of SGLT2i in people with T2D is recommended in previous guidelines

In [11]:
# Sanity checks: the statements the first attempt missed
for sid in ["Rec 3.4.1", "Rec 3.6.1", "Rec 3.7.2", "PP 2.1.1", "Rec 3.15.1.1", "PP 1.3.1.4", "PP 5.3.1.1.1"]:
    s = next(x for x in statements if x["statement_id"] == sid)
    print(f"{sid:14} p{s['page']:<4} [{s['heading_path']}]\n    {s['text'][:200]}\n")

Rec 3.4.1      p97   [Chapter 3: Delaying CKD progression and managing its complications › 3.4 Blood pressure control]
    Recommendation 3.4.1: We suggest that adults with high BP and CKD be treated with a target systolic blood pressure (SBP) of <120 mm Hg, when tolerated, using standardized office BP measurement (2B).

Rec 3.6.1      p98   [Chapter 3: Delaying CKD progression and managing its complications › 3.6 Renin-angiotensin system inhibitors]
    Recommendation 3.6.1: We recommend starting renin-angiotensin-system inhibitors (RASi) (angiotensin-converting enzyme inhibitor [ACEi] or angiotensin II receptor blocker [ARB]) for people with CKD and

Rec 3.7.2      p99   [Chapter 3: Delaying CKD progression and managing its complications › 3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i)]
    Recommendation 3.7.2: We recommend treating adults with CKD with an SGLT2i for the following (1A):
 eGFR ≥20 ml/min per 1.73 m2 with urine ACR ≥200 mg/g (≥20 mg/mmol), or
 heart failure,

## 4. Tables

Captioned tables become their own retrievable units (markdown). Figures are images — only
their captions are indexed. The core KDIGO lookups (GFR/albuminuria categories, risk heat map)
are encoded **as code** in §6 instead of being retrieved.

In [12]:
table_units = {}
for i, item in enumerate(items[:BODY_END]):
    if not isinstance(item, TableItem):
        continue
    caption = clean_text(item.caption_text(doc) or "")
    m = re.match(r"^(Table|Figure)\s*(\d+)\s*\|?\s*(.*)", caption)
    if not m:
        continue
    key = f"{m.group(1)} {m.group(2)}"
    markdown = clean_text(item.export_to_markdown(doc))
    # keep the largest rendering if a table is repeated / continued
    if key not in table_units or len(markdown) > len(table_units[key]["markdown"]):
        table_units[key] = {"table_id": key, "caption": caption, "markdown": markdown, "page": page_of(item)}

# Figures that are images: keep caption only (e.g. Figure 55, heat maps)
for item in items[:BODY_END]:
    if isinstance(item, PictureItem):
        caption = clean_text(item.caption_text(doc) or "")
        m = re.match(r"^(Figure)\s*(\d+)\s*\|?\s*(.*)", caption)
        if m and f"Figure {m.group(2)}" not in table_units:
            key = f"Figure {m.group(2)}"
            table_units[key] = {"table_id": key, "caption": caption, "markdown": "", "page": page_of(item)}

print(len(table_units), "captioned tables/figures")
print(sorted(table_units, key=lambda k: (k.split()[0], int(k.split()[1]))))
print(table_units["Table 2"]["markdown"])

85 captioned tables/figures
['Figure 1', 'Figure 2', 'Figure 3', 'Figure 4', 'Figure 5', 'Figure 6', 'Figure 7', 'Figure 8', 'Figure 10', 'Figure 11', 'Figure 12', 'Figure 13', 'Figure 14', 'Figure 15', 'Figure 16', 'Figure 17', 'Figure 18', 'Figure 19', 'Figure 21', 'Figure 22', 'Figure 23', 'Figure 24', 'Figure 25', 'Figure 27', 'Figure 28', 'Figure 29', 'Figure 30', 'Figure 31', 'Figure 32', 'Figure 34', 'Figure 35', 'Figure 37', 'Figure 38', 'Figure 41', 'Figure 42', 'Figure 43', 'Figure 45', 'Figure 47', 'Figure 48', 'Figure 49', 'Figure 50', 'Figure 51', 'Figure 52', 'Figure 53', 'Figure 54', 'Figure 55', 'Figure 56', 'Table 1', 'Table 2', 'Table 3', 'Table 4', 'Table 5', 'Table 6', 'Table 7', 'Table 8', 'Table 9', 'Table 12', 'Table 13', 'Table 14', 'Table 15', 'Table 16', 'Table 18', 'Table 19', 'Table 20', 'Table 21', 'Table 22', 'Table 23', 'Table 24', 'Table 25', 'Table 26', 'Table 27', 'Table 28', 'Table 29', 'Table 30', 'Table 31', 'Table 32', 'Table 33', 'Table 34', 'Tabl

In [13]:
# How many statements reference a table we captured?
refs = Counter(r for s in statements for r in s["table_refs"])
print({r: (c, r in table_units) for r, c in refs.most_common()})

{'Figure 55': (4, True), 'Figure 8': (1, True), 'Table 6': (1, True), 'Table 8': (1, True), 'Figure 11': (1, True), 'Table 9': (1, True), 'Table 11': (1, False), 'Table 16': (1, True), 'Table 17': (1, False), 'Figure 17': (1, True), 'Figure 26': (1, False), 'Figure 40': (1, False), 'Figure 44': (1, False), 'Figure 48': (1, True), 'Table 41': (1, False)}


## 5. Applicability metadata — deterministic first

Numeric thresholds are extracted with regex (reliable once glyphs are fixed); the LLM is
only an optional second pass (§5b), and a human reviews the CSV (§5c).

Each statement gets an **applicability** record. Constraints are used for **exclusion only**:

| field | example | used how |
|---|---|---|
| `population` | `child` / `adult` / `both` / `any` | drop child-only statements for adults and vice versa |
| `egfr_range` | `[0, 60)` from "eGFR <60" or "G3a-G5" | drop if patient eGFR known and outside |
| `acr_range` (mg/g) | `[200, inf)` | drop if ACR known and outside; **flag "needs ACR"** if unknown |
| `age_range` | `[50, inf)` | drop if age outside |
| `diabetes` | `required` / `excluded` | drop if contradicted; flag if unknown |
| `excludes_krt` | "not treated with chronic dialysis or kidney transplantation" | drop for dialysis/transplant |
| `requires_krt` | "people receiving dialysis" | drop if patient is not on KRT |
| `drug_classes` | `SGLT2i`, `RASi` | routing / boosting, never exclusion |

Ranges from several mentions are combined as the **hull** (min lower, max upper) — conservative:
a statement with "G4-G5 **or** aged >65" won't be wrongly dropped for one branch.

In [14]:
INF = math.inf
G_RANGES = {"G1": (90, INF), "G2": (60, 90), "G3a": (45, 60), "G3b": (30, 45), "G3": (30, 60), "G4": (15, 30), "G5": (0, 15)}
A_RANGES = {"A1": (0, 30), "A2": (30, 300), "A3": (300, INF)}  # mg/g


def interval_from_op(op, value):
    value = float(value)
    return {"<": (0, value), "≤": (0, value + 1e-9), ">": (value + 1e-9, INF), "≥": (value, INF)}[op]


def hull(intervals):
    if not intervals:
        return None
    return (min(lo for lo, _ in intervals), max(hi for _, hi in intervals))


CHILD_RE = re.compile(r"\b(child(?![- ]bearing)|children|pediatric|paediatric|infant|neonat\w*|adolescen\w*|young people)\b", re.I)
ADULT_RE = re.compile(r"\badults?\b(?!-oriented| care| provider| clinic)", re.I)
G_RANGE_RE = re.compile(r"\bG(1|2|3a|3b|3|4|5)\s*[-–]\s*G?(1|2|3a|3b|3|4|5)\b")
G_SINGLE_RE = re.compile(r"\b(?:CKD|categor(?:y|ies))\s+G(1|2|3a|3b|3|4|5)\b(?!\s*[-–])")
EGFR_OP_RE = re.compile(r"\beGFR\w*\s*(?:of\s*|level\s*)?([<>≥≤])\s*(\d+)")
EGFR_BETWEEN_RE = re.compile(r"\beGFR\w*\s*(?:of\s*)?(\d+)\s*[-–]\s*(\d+)")
ACR_OP_RE = re.compile(r"([<>≥≤])\s*(\d+)\s*mg/g")
A_RANGE_RE = re.compile(r"\bA([123])\s*[-–]\s*A?([123])\b|\b[G\d]\w*\s*[-–]?\s*\w*,\s*A([123])\b")
AGE_OP_RE = re.compile(r"\baged?\s*([<>≥≤])\s*(\d+)")
AGE_BETWEEN_RE = re.compile(r"\baged?\s*(\d+)\s*[-–]\s*(\d+)")
DIABETES_REQ_RE = re.compile(r"\b(?:with|and)\s+(?:type 2 diabetes|T2D|diabetes)\b", re.I)
DIABETES_EXCL_RE = re.compile(r"\bwithout diabetes\b", re.I)
EXCLUDES_KRT_RE = re.compile(r"not (?:treated with|receiving|on) (?:chronic )?(?:dialysis|KRT)", re.I)
REQUIRES_KRT_RE = re.compile(r"\b(?:receiving|on|treated with) (?:chronic |maintenance )?(?:dialysis|KRT)\b|\bkidney transplant recipients\b", re.I)

DRUG_CLASS_PATTERNS = {
    "RASi": r"\bRASi\b|\bACEi\b|\bARBs?\b|renin[- ]angiotensin|angiotensin",
    "SGLT2i": r"SGLT2",
    "MRA": r"mineralocorticoid|finerenone|\bMRA\b",
    "GLP-1 RA": r"GLP-1",
    "statin": r"statin",
    "metformin": r"metformin",
    "NSAID": r"NSAID|nonsteroidal anti-inflammatory",
    "potassium binder": r"potassium[- ]binder|potassium exchange",
    "urate-lowering": r"urate-lowering|allopurinol|febuxostat",
    "anticoagulant": r"anticoagula|DOAC|warfarin|NOAC",
    "bicarbonate": r"bicarbonate",
}


def extract_applicability(s):
    text = s["text"]
    context = f"{s['heading_path']} {text}"
    meta = {}

    child = bool(CHILD_RE.search(context))
    adult = bool(ADULT_RE.search(text))
    meta["population"] = "both" if child and adult else "child" if child else "adult" if adult else "any"

    egfr = []
    for a, b in G_RANGE_RE.findall(text):
        lo_a, hi_a = G_RANGES[f"G{a}"]
        lo_b, hi_b = G_RANGES[f"G{b}"]
        egfr.append((min(lo_a, lo_b), max(hi_a, hi_b)))
    for g in G_SINGLE_RE.findall(text):
        egfr.append(G_RANGES[f"G{g}"])
    egfr += [interval_from_op(op, v) for op, v in EGFR_OP_RE.findall(text)]
    egfr += [(float(a), float(b) + 1) for a, b in EGFR_BETWEEN_RE.findall(text)]
    meta["egfr_range"] = hull(egfr)

    acr = [interval_from_op(op, v) for op, v in ACR_OP_RE.findall(text)]
    for a, b, single in A_RANGE_RE.findall(text):
        if single:
            acr.append(A_RANGES[f"A{single}"])
        else:
            acr.append((A_RANGES[f"A{a}"][0], A_RANGES[f"A{b}"][1]))
    if re.search(r"severely increased albuminuria", text, re.I):
        acr.append(A_RANGES["A3"])
    if re.search(r"moderately increased albuminuria", text, re.I):
        acr.append(A_RANGES["A2"])
    meta["acr_range"] = hull(acr)

    age = [interval_from_op(op, v) for op, v in AGE_OP_RE.findall(text)]
    age += [(float(a), float(b) + 1) for a, b in AGE_BETWEEN_RE.findall(text)]
    meta["age_range"] = hull(age)

    meta["diabetes"] = "excluded" if DIABETES_EXCL_RE.search(text) else "required" if DIABETES_REQ_RE.search(text) else None
    meta["excludes_krt"] = bool(EXCLUDES_KRT_RE.search(text))
    meta["requires_krt"] = bool(REQUIRES_KRT_RE.search(text)) and not meta["excludes_krt"]
    meta["mentions_heart_failure"] = bool(re.search(r"heart failure", text, re.I))
    meta["mentions_pregnancy"] = bool(re.search(r"pregnan", text, re.I))
    meta["drug_classes"] = [k for k, pat in DRUG_CLASS_PATTERNS.items() if re.search(pat, text, re.I)]
    meta["metadata_source"] = "regex"
    return meta


for s in statements:
    s["applicability"] = extract_applicability(s)

for sid in ["Rec 3.15.1.1", "Rec 3.7.1", "Rec 3.7.2", "Rec 3.6.1", "Rec 3.8.1", "PP 1.2.3.5", "PP 5.2.3.2", "PP 1.3.1.4"]:
    s = next(x for x in statements if x["statement_id"] == sid)
    print(f"{sid:13}", {k: v for k, v in s["applicability"].items() if v not in (None, False, [], "any", "regex")})

Rec 3.15.1.1  {'population': 'adult', 'egfr_range': (0, 60), 'age_range': (50.0, inf), 'excludes_krt': True, 'drug_classes': ['statin']}
Rec 3.7.1     {'egfr_range': (20.0, inf), 'diabetes': 'required', 'drug_classes': ['SGLT2i']}
Rec 3.7.2     {'population': 'adult', 'egfr_range': (20.0, inf), 'acr_range': (200.0, inf), 'mentions_heart_failure': True, 'drug_classes': ['SGLT2i']}
Rec 3.6.1     {'acr_range': (300, inf), 'diabetes': 'excluded', 'drug_classes': ['RASi']}
Rec 3.8.1     {'population': 'adult', 'egfr_range': (25.000000001, inf), 'diabetes': 'required', 'drug_classes': ['RASi', 'MRA']}
PP 1.2.3.5    {'population': 'child', 'egfr_range': (0, 90.0)}
PP 5.2.3.2    {'egfr_range': (0, 30), 'age_range': (65.000000001, inf)}
PP 1.3.1.4    {'population': 'child'}


### 5a. Manual overrides

Regex can't read intent. Where a clinician review finds an error, record it here — overrides
are data, versioned with the index, and always win.

In [15]:
MANUAL_OVERRIDES = {
    # Transition-of-care statements are about adolescents / young adults, not all adults.
    # (Illustrative — to be confirmed by clinical review.)
    **{sid: {"population": "child"} for sid in ["PP 5.3.1.1.1", "PP 5.3.1.1.2", "PP 5.3.1.1.3"]},
    **{sid: {"age_range": (0, 26)} for sid in ["PP 5.3.1.2.1", "PP 5.3.1.2.2", "PP 5.3.1.2.3"]},
    # "A or B" statements: the filter can only express AND, so drop the constraint rather than exclude
    # people who qualify through the other branch (found by the §11 filter-error check).
    "Rec 3.7.2": {"acr_range": None},                     # "... ACR >=200, or heart failure irrespective of albuminuria"
    "PP 5.2.3.2": {"egfr_range": None, "age_range": None},  # "G4-G5, aged >65, poor growth, or symptoms such as weight loss"
}

for s in statements:
    if s["statement_id"] in MANUAL_OVERRIDES:
        s["applicability"].update(MANUAL_OVERRIDES[s["statement_id"]])
        s["applicability"]["metadata_source"] = "manual"

print(Counter(s["applicability"]["population"] for s in statements))
print("with eGFR constraint:", sum(s["applicability"]["egfr_range"] is not None for s in statements))
print("with ACR constraint:", sum(s["applicability"]["acr_range"] is not None for s in statements))
print("with age constraint:", sum(s["applicability"]["age_range"] is not None for s in statements))

Counter({'any': 126, 'adult': 20, 'child': 19, 'both': 4})
with eGFR constraint: 19
with ACR constraint: 6
with age constraint: 7


### 5b. LLM applicability, second opinion (gated by `RUN_LLM`)

The LLM reads each statement (constrained to a schema, quoting the supporting span) and its answer
is stored as a **second, independent** applicability record (`applicability_llm`).

Tested on the gold set (§11), neither source is safe alone as a hard filter: regex turns "A **or** B"
into "A required", and the LLM invents constraints (e.g. "diabetes required" on generic statements).
So the filter (§8) uses **consensus**: a statement is excluded only when *both* records exclude it.

Every regex/LLM disagreement is written to `outputs/applicability_llm_disagreements.csv` and shown in the
`llm_differs` column of the review sheet (§5c): those are the rows to review first.


In [16]:
class LLMApplicability(BaseModel):
    population: Literal["adult", "child", "both", "any"]
    population_quote: Optional[str] = Field(None, description="Exact span from the text supporting population, or null")
    egfr_min: Optional[float] = Field(None, description="Inclusive lower eGFR bound, ml/min/1.73m2")
    egfr_max: Optional[float] = Field(None, description="Exclusive upper eGFR bound, ml/min/1.73m2")
    egfr_quote: Optional[str] = None
    acr_min: Optional[float] = Field(None, description="Inclusive lower urine ACR bound, mg/g")
    acr_max: Optional[float] = Field(None, description="Exclusive upper urine ACR bound, mg/g")
    acr_quote: Optional[str] = None
    age_min: Optional[float] = Field(None, description="Inclusive lower age bound, years")
    age_max: Optional[float] = Field(None, description="Exclusive upper age bound, years")
    age_quote: Optional[str] = None
    requires_conditions: list[Literal["type_2_diabetes", "diabetes", "heart_failure", "hypertension", "dialysis", "kidney_transplant", "pregnancy", "albuminuria"]] = []
    excludes_conditions: list[Literal["diabetes", "dialysis", "kidney_transplant", "pregnancy"]] = []
    conditions_quote: Optional[str] = None
    intent: Literal["diagnosis_evaluation", "monitoring", "treatment_start", "treatment_continue_or_stop", "referral", "education_or_care_model", "laboratory_practice"]


LLM_META_SYSTEM = (
    "You extract applicability criteria from ONE clinical guideline statement (KDIGO 2024 CKD). "
    "A criterion is a condition the PATIENT must meet for the statement to apply. "
    "Use only what is explicitly stated in the statement text or its heading path. "
    "If a criterion is not stated, leave it null / empty / 'any'. Quotes must be verbatim spans.\n"
    "- population: 'adult' or 'child' only if the statement is restricted to that group; "
    "'both' if it explicitly addresses adults and children; otherwise 'any'.\n"
    "- Ranges: min is inclusive, max is exclusive. 'eGFR <60' -> egfr_max=60; 'eGFR >=20' -> egfr_min=20; "
    "'eGFR 20 to 45' -> egfr_min=20, egfr_max=46. GFR categories: G1 >=90, G2 60-89, G3a 45-59, G3b 30-44, "
    "G4 15-29, G5 <15 (so 'G1-G4' -> egfr_min=15). Albuminuria: A1 <30, A2 30-299, A3 >=300 mg/g.\n"
    "- A number that only describes a result to act on (e.g. 'a change in eGFR of >20%', 'serum bicarbonate <18') "
    "is NOT a patient eligibility range.\n"
    "- 'with or without diabetes' means diabetes is neither required nor excluded.\n"
    "- Alternatives joined by 'or' (any ONE qualifies, e.g. 'eGFR >=20 with ACR >=200, or heart failure'; "
    "'CKD G4-G5, aged >65, or weight loss'): do NOT turn one alternative into a range or condition — leave it null.\n"
    "- A threshold that says WHEN to do something for people already covered (e.g. 'plan dialysis access when GFR <15-20', "
    "'caution at G4-G5', 'the eGFR at which transplantation is undertaken', 'continue even when eGFR falls below 30') "
    "is NOT an eligibility range.\n"
    "- dialysis / kidney_transplant go in requires_conditions only if the statement is about people who are ALREADY on "
    "dialysis or have a transplant — not when it is about informing, planning or deciding to start KRT.\n"
    "- A range with no upper limit (e.g. 'G1-G4', 'eGFR >=20') has egfr_max = null."
)
LLM_META_VERSION = hashlib.sha1(LLM_META_SYSTEM.encode()).hexdigest()[:8]  # prompt change -> fresh LLM calls


def llm_applicability(s):
    from openai import OpenAI
    client = OpenAI(max_retries=10)
    resp = client.responses.parse(
        model=LLM_MODEL,
        input=[
            {"role": "system", "content": LLM_META_SYSTEM},
            {"role": "user", "content": f"Heading path: {s['heading_path']}\n\nStatement:\n{s['text']}"},
        ],
        text_format=LLMApplicability,
    )
    return resp.output_parsed.model_dump()


def _llm_range(llm, name):
    lo, hi = llm[f"{name}_min"], llm[f"{name}_max"]
    return None if lo is None and hi is None else (lo if lo is not None else 0, hi if hi is not None else INF)


def merge_llm_applicability(rx, llm):
    req, excl = set(llm["requires_conditions"]), set(llm["excludes_conditions"])
    merged = dict(rx)
    merged["population"] = llm["population"]
    merged["diabetes"] = "excluded" if "diabetes" in excl else "required" if req & {"diabetes", "type_2_diabetes"} else None
    merged["excludes_krt"] = bool(excl & {"dialysis", "kidney_transplant"})
    merged["requires_krt"] = bool(req & {"dialysis", "kidney_transplant"})
    for name in ("egfr", "acr", "age"):
        merged[f"{name}_range"] = _llm_range(llm, name)
    merged["metadata_source"] = "llm"
    return merged


if RUN_LLM:
    from concurrent.futures import ThreadPoolExecutor

    todo = [s for s in statements if s["applicability"]["metadata_source"] != "manual"]
    with ThreadPoolExecutor(max_workers=3) as pool:
        llm_answers = list(pool.map(
            lambda s: llm_cached(f"applicability|{LLM_META_VERSION}|{s['statement_id']}|{s['text']}", lambda: llm_applicability(s)), todo))

    disagreements = []
    for s, llm in zip(todo, llm_answers):
        s["llm_applicability"] = llm
        rx = s["applicability"]
        merged = merge_llm_applicability(rx, llm)
        changed = {k: (rx[k], merged[k]) for k in ("population", "diabetes", "excludes_krt", "requires_krt", "egfr_range", "acr_range", "age_range") if rx[k] != merged[k]}
        if changed:
            disagreements.append({
                "statement_id": s["statement_id"],
                "regex -> llm": "; ".join(f"{k}: {a} -> {b}" for k, (a, b) in changed.items()),
                "llm_quotes": " | ".join(q for q in (llm["population_quote"], llm["egfr_quote"], llm["acr_quote"], llm["age_quote"], llm["conditions_quote"]) if q),
                "text": s["text"][:200],
            })
        s["applicability_llm"] = merged

    disagreements_df = pd.DataFrame(disagreements)
    disagreements_df.to_csv(OUTPUT_DIR / "applicability_llm_disagreements.csv", index=False, encoding="utf-8-sig")
    print(f"LLM applicability merged for {len(todo)} statements; {len(disagreements)} differ from regex")
    llm_disagreement_by_id = dict(zip(disagreements_df["statement_id"], disagreements_df["regex -> llm"])) if len(disagreements_df) else {}
    display(disagreements_df)
else:
    llm_disagreement_by_id = {}
    print("RUN_LLM=False — skipped LLM enrichment")


LLM applicability merged for 161 statements; 32 differ from regex


,statement_id,regex -> llm,llm_quotes,text
0,PP 1.1.4.1,diabetes: None -> required,cause of CKD,"Practice Point 1.1.4.1: Establish the cause of CKD using clinical context, personal and family history, social and e..."
1,PP 1.2.3.5,"age_range: None -> (2.0, inf)",children and adolescents over the age of 2 years | eGFRcr level <90ml/min per 1.73m 2 | over the age of 2 years,Practice Point 1.2.3.5: An eGFRcr level <90ml/min per 1.73m 2 can be flagged as 'low' in children and adolescents ov...
2,PP 1.2.4.1,population: both -> any,,"Practice Point 1.2.4.1: Use the same equation within geographical regions (as defined locally [e.g., continent, coun..."
3,PP 1.3.1.1,population: both -> any,"In all cases, a first void in the morning midstream sample is preferred in adults and children.",Practice Point 1.3.1.1: Use the following measurements for initial testing of albuminuria (in descending order of pr...
4,PP 1.3.1.2,"acr_range: (30.0, inf) -> None",,Practice Point 1.3.1.2: Use more accurate methods when albuminuria is detected using less accurate methods.\n Confi...
5,PP 1.4.3,"acr_range: (30.0, inf) -> None",,"Practice Point 1.4.3: Where a POCT device is being used for albuminuria testing, the capability of also analyzing cr..."
6,PP 2.1.1,population: both -> any,"adults, or albuminuria/proteinuria in children | people with CKD","Practice Point 2.1.1: Assess albuminuria in adults, or albuminuria/proteinuria in children, and GFR at least annuall..."
7,PP 2.1.3,"egfr_range: (20.000000001, inf) -> None",For people with CKD,"Practice Point 2.1.3: For people with CKD, a change in eGFR of >20% on a subsequent test exceeds the expected variab..."
8,Rec 2.2.1,"egfr_range: (0, 60) -> (30.0, 60.0)",CKD G3-G5,"Recommendation 2.2.1: In people with CKD G3- G5, we recommend using an externally validated risk equation to estimat..."
9,PP 2.2.4,"egfr_range: (0, inf) -> None","CKD G3-G5, may not be valid for use in those with CKD G1-G2","Practice Point 2.2.4: Note that risk prediction equations developed for use in people with CKD G3-G5, may not be val..."


### 5c. Clinician review of applicability (spreadsheet round-trip)

1. This cell writes `outputs/statement_applicability_review.csv` (the machine's current answers).
2. The clinician opens it in Excel, **edits wrong values in place**, types `Y` in `reviewed`,
   and saves as `outputs/statement_applicability_reviewed.csv` (new name, so re-running never
   overwrites the review).
3. The next cell loads the reviewed file (if present) and applies every row marked `Y`.
   Clinician values always win over regex and LLM.

Column meanings — each is a condition a patient must meet for the statement to apply
(a blank cell means "no limit / not specified"):

| column | values |
|---|---|
| `population` | `adult`, `child`, `both`, `any` |
| `egfr_min`, `egfr_max` | ml/min/1.73m² — applies if `min ≤ eGFR < max` |
| `acr_min`, `acr_max` | mg/g — same rule |
| `age_min`, `age_max` | years — same rule |
| `diabetes` | `required`, `excluded`, blank |
| `excludes_krt` | `True` if not for dialysis / transplant patients |
| `requires_krt` | `True` if only for dialysis / transplant patients |

In [17]:
RANGE_FIELDS = {"egfr": "egfr_range", "acr": "acr_range", "age": "age_range"}
REVIEW_EXPORT = OUTPUT_DIR / "statement_applicability_review.csv"
REVIEW_IMPORT = OUTPUT_DIR / "statement_applicability_reviewed.csv"


def _fmt(v):
    return "" if v is None or v == INF else round(v, 2)


review_df = pd.DataFrame([
    {
        "statement_id": s["statement_id"],
        "grade": s["grade"],
        "domain": s["domain"],
        "text": s["text"],
        "population": s["applicability"]["population"],
        **{f"{name}_{end}": _fmt(s["applicability"][key][i]) if s["applicability"][key] else ""
           for name, key in RANGE_FIELDS.items() for i, end in enumerate(["min", "max"])},
        "diabetes": s["applicability"]["diabetes"] or "",
        "excludes_krt": s["applicability"]["excludes_krt"],
        "requires_krt": s["applicability"]["requires_krt"],
        "machine_source": s["applicability"]["metadata_source"],
        "llm_differs": llm_disagreement_by_id.get(s["statement_id"], ""),
        "reviewed": "",
        "reviewer_notes": "",
    }
    for s in statements
])
review_df.to_csv(REVIEW_EXPORT, index=False, encoding="utf-8-sig")  # utf-8-sig so Excel shows ≥ correctly
print("wrote", REVIEW_EXPORT, len(review_df), "rows")
review_df[review_df["population"] != "any"][["statement_id", "population", "egfr_min", "egfr_max", "age_min", "text"]].head(15)

wrote outputs/statement_applicability_review.csv 169 rows


,statement_id,population,egfr_min,egfr_max,age_min,text
2,Rec 1.1.2.1,adult,,,,"Recommendation 1.1.2.1: In adults at risk for CKD, we recommend using creatinine-based estimated glomerular filtrati..."
21,PP 1.2.3.3,child,,,,Practice Point 1.2.3.3: Laboratories measuring creatinine in infants or small children must ensure their quality con...
22,PP 1.2.3.4,child,,,,"Practice Point 1.2.3.4: Consider the consistent use of enzymatic creatinine assays in children, given the higher rel..."
23,PP 1.2.3.5,child,0,90.0,,Practice Point 1.2.3.5: An eGFRcr level <90ml/min per 1.73m 2 can be flagged as 'low' in children and adolescents ov...
25,PP 1.2.4.1,both,,,,"Practice Point 1.2.4.1: Use the same equation within geographical regions (as defined locally [e.g., continent, coun..."
27,PP 1.2.4.3,child,,,,Practice Point 1.2.4.3: Estimate GFR in children using validated equations that have been developed or validated in ...
28,PP 1.3.1.1,both,,,,Practice Point 1.3.1.1: Use the following measurements for initial testing of albuminuria (in descending order of pr...
31,PP 1.3.1.4,child,,,,"Practice Point 1.3.1.4: In children, obtain a first morning urine sample for initial testing of albuminuria and prot..."
38,PP 2.1.1,both,,,,"Practice Point 2.1.1: Assess albuminuria in adults, or albuminuria/proteinuria in children, and GFR at least annuall..."
58,PP 3.2.2.5,child,,,,Practice Point 3.2.2.5: Encourage children with CKD to undertake physical activity aiming for World Health Organizat...


In [18]:
def read_review_csv(path):
    # Tolerates Excel and Apple Numbers exports: optional BOM, an optional table-name line
    # above the header (Numbers "Include table names"), and trailing empty rows.
    raw = Path(path).read_bytes().decode("utf-8-sig", errors="replace").splitlines()
    header_row = next(i for i, line in enumerate(raw) if "statement_id" in line)
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str, skiprows=header_row, encoding_errors="replace")
    df.columns = [c.strip() for c in df.columns]
    return df[df["statement_id"].notna()].fillna("")


def _num(v, default):
    if v is None or (isinstance(v, float) and math.isnan(v)) or str(v).strip() == "":
        return default
    return float(v)


def _bool(v):
    return str(v).strip().lower() in ("true", "1", "y", "yes")


def apply_clinician_review(path):
    reviewed = read_review_csv(path)
    reviewed = reviewed[reviewed["reviewed"].str.strip().str.upper() == "Y"]
    changes = []
    for _, row in reviewed.iterrows():
        s = stmt_by_id_all.get(row["statement_id"])
        if s is None:
            print("unknown statement_id in review file:", row["statement_id"])
            continue
        new = {
            "population": row["population"].strip() or "any",
            "diabetes": row["diabetes"].strip() or None,
            "excludes_krt": _bool(row["excludes_krt"]),
            "requires_krt": _bool(row["requires_krt"]),
        }
        for name, key in RANGE_FIELDS.items():
            lo, hi = _num(row[f"{name}_min"], None), _num(row[f"{name}_max"], None)
            new[key] = None if lo is None and hi is None else (lo if lo is not None else 0, hi if hi is not None else INF)
        before = {k: s["applicability"][k] for k in new}
        if before != new:
            changes.append({"statement_id": row["statement_id"], "before": before, "after": new, "notes": row["reviewer_notes"]})
        s["applicability"].update(new)
        s["applicability"]["metadata_source"] = "clinician"
        s.pop("applicability_llm", None)  # clinician answer replaces both machine opinions
    return reviewed, changes


stmt_by_id_all = {s["statement_id"]: s for s in statements}

if REVIEW_IMPORT.exists():
    reviewed_rows, review_changes = apply_clinician_review(REVIEW_IMPORT)
    print(f"applied clinician review: {len(reviewed_rows)} rows reviewed, {len(review_changes)} corrected")
    display(pd.DataFrame(review_changes))
else:
    print(f"no {REVIEW_IMPORT.name} yet — using machine-extracted metadata")

no statement_applicability_reviewed.csv yet — using machine-extracted metadata


## 6. Deterministic KDIGO lookups (encoded, not retrieved)

GFR / albuminuria categories and the risk heat map are small lookup tables. Retrieving them
from a PDF figure is unreliable; encoding them is exact and testable.

In [19]:
def gfr_category(egfr):
    if egfr is None:
        return None
    for cat, (lo, hi) in G_RANGES.items():
        if cat != "G3" and lo <= egfr < hi:
            return cat


def acr_category(acr_mg_g):
    if acr_mg_g is None:
        return None
    for cat, (lo, hi) in A_RANGES.items():
        if lo <= acr_mg_g < hi:
            return cat


# KDIGO heat map (Figure: prognosis of CKD by GFR and albuminuria categories)
RISK_HEATMAP = {
    "G1": {"A1": "low", "A2": "moderate", "A3": "high"},
    "G2": {"A1": "low", "A2": "moderate", "A3": "high"},
    "G3a": {"A1": "moderate", "A2": "high", "A3": "very high"},
    "G3b": {"A1": "high", "A2": "very high", "A3": "very high"},
    "G4": {"A1": "very high", "A2": "very high", "A3": "very high"},
    "G5": {"A1": "very high", "A2": "very high", "A3": "very high"},
}

assert gfr_category(48) == "G3a" and gfr_category(38) == "G3b" and gfr_category(94) == "G1"
assert acr_category(250) == "A2"
print(gfr_category(48), RISK_HEATMAP["G3a"])

G3a {'A1': 'moderate', 'A2': 'high', 'A3': 'very high'}


## 7. Patient state

Two sources, merged:

1. **Structured history** (EHR record: labs, diagnoses, medications) — deterministic mapping.
2. **Clinician's free-text note for the current visit** — LLM extraction into the same schema
   (gated by `RUN_LLM`).

Merge policy: structured labs win over values mentioned in the note unless the note is newer;
the note adds symptoms, BP readings and anything new. Then **derived fields are computed in
code** (G category, adult, drug classes, missing data) — never by the LLM.

In [20]:
class PatientState(BaseModel):
    age: Optional[int] = None
    sex: Optional[Literal["M", "F"]] = None
    egfr: Optional[float] = None
    egfr_date: Optional[str] = None
    acr_mg_g: Optional[float] = None
    potassium: Optional[float] = None
    sbp: Optional[int] = None
    known_ckd: Optional[bool] = None
    hypertension: Optional[bool] = None
    diabetes: Literal["type_1", "type_2", "none", "unknown"] = "unknown"
    heart_failure: Optional[bool] = None
    on_dialysis: Optional[bool] = None
    kidney_transplant: Optional[bool] = None
    pregnant: Optional[bool] = None
    medications: list[str] = []
    symptoms: list[str] = []
    concerns: list[str] = Field(default_factory=list, description="Clinician's stated concerns / questions this visit")


DRUG_TO_CLASS = {
    r"pril\b": "RASi", r"sartan\b": "RASi",
    r"gliflozin\b": "SGLT2i",
    r"finerenone|spironolactone|eplerenone": "MRA",
    r"glutide\b|tirzepatide": "GLP-1 RA",
    r"statin\b": "statin",
    r"amlodipine|nifedipine|felodipine": "CCB",
    r"metformin": "metformin",
    r"ibuprofen|naproxen|diclofenac": "NSAID",
}


def drug_classes(medications):
    return sorted({cls for med in medications for pat, cls in DRUG_TO_CLASS.items() if re.search(pat, med, re.I)})


def derive(state: PatientState) -> dict:
    missing = []
    if state.acr_mg_g is None:
        missing.append("urine ACR")
    if state.diabetes == "unknown":
        missing.append("diabetes status")
    if state.egfr is None:
        missing.append("eGFR")
    g, a = gfr_category(state.egfr), acr_category(state.acr_mg_g)
    return {
        "is_adult": state.age is None or state.age >= 18,
        "gfr_category": g,
        "acr_category": a,
        "risk_category": RISK_HEATMAP.get(g, {}).get(a) if g and a else None,
        "drug_classes": drug_classes(state.medications),
        "missing_data": missing,
    }

In [21]:
# Synthetic patients. P001-P003 are the ones from initial_experiment.ipynb; P004-P012 are test cases
# chosen to exercise one guideline area / filter rule each.
patients = {
    "P001": {
        "demographics": {"age": 67, "sex": "M"},
        "diagnoses": ["Hypertension"],
        "medications": ["Amlodipine 10 mg daily"],
        "labs": [
            {"date": "2026-04-12", "test": "eGFR", "value": 48, "unit": "mL/min/1.73m2"},
            {"date": "2026-04-12", "test": "Creatinine", "value": 1.6, "unit": "mg/dL"},
        ],
    },
    "P002": {
        "demographics": {"age": 54, "sex": "F"},
        "diagnoses": ["Hypertension", "Chronic kidney disease"],
        "medications": ["Ramipril 10 mg daily"],
        "labs": [
            {"date": "2026-03-10", "test": "eGFR", "value": 38, "unit": "mL/min/1.73m2"},
            {"date": "2026-03-10", "test": "Creatinine", "value": 1.8, "unit": "mg/dL"},
        ],
    },
    "P003": {
        "demographics": {"age": 42, "sex": "M"},
        "diagnoses": [],
        "medications": [],
        "labs": [
            {"date": "2026-05-02", "test": "eGFR", "value": 94, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-02", "test": "Creatinine", "value": 0.9, "unit": "mg/dL"},
        ],
    },
    # T2D + severely increased albuminuria on ACEi + metformin: SGLT2i, finerenone, GLP-1 RA, referral (ACR >=300)
    "P004": {
        "demographics": {"age": 61, "sex": "F"},
        "diagnoses": ["Type 2 diabetes", "Hypertension", "Chronic kidney disease"],
        "medications": ["Metformin 1 g twice daily", "Lisinopril 40 mg daily"],
        "labs": [
            {"date": "2026-02-20", "test": "eGFR", "value": 55, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-18", "test": "eGFR", "value": 52, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-18", "test": "ACR", "value": 420, "unit": "mg/g"},
            {"date": "2026-05-18", "test": "Potassium", "value": 4.6, "unit": "mmol/L"},
            {"date": "2026-05-18", "test": "HbA1c", "value": 8.2, "unit": "%"},
        ],
    },
    # Advanced CKD G4 A3 with hyperkalemia on ARB and uremic symptoms: KRT planning, referral, potassium
    "P005": {
        "demographics": {"age": 72, "sex": "M"},
        "diagnoses": ["Hypertension", "Chronic kidney disease"],
        "medications": ["Losartan 100 mg daily", "Amlodipine 10 mg daily"],
        "labs": [
            {"date": "2025-11-02", "test": "eGFR", "value": 27, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-30", "test": "eGFR", "value": 22, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-30", "test": "ACR", "value": 850, "unit": "mg/g"},
            {"date": "2026-05-30", "test": "Potassium", "value": 5.6, "unit": "mmol/L"},
        ],
    },
    # Young woman, new A2 albuminuria + microscopic hematuria, no diabetes, untreated high BP: cause, RASi, teratogenicity
    "P006": {
        "demographics": {"age": 45, "sex": "F"},
        "diagnoses": [],
        "medications": [],
        "labs": [
            {"date": "2026-06-01", "test": "eGFR", "value": 68, "unit": "mL/min/1.73m2"},
            {"date": "2026-06-01", "test": "ACR", "value": 150, "unit": "mg/g"},
        ],
    },
    # Child: CAKUT, CKD G3a, high BP — pediatric statements must appear, adult-only must not
    "P007": {
        "demographics": {"age": 12, "sex": "M"},
        "diagnoses": ["Chronic kidney disease", "Congenital anomalies of the kidney and urinary tract"],
        "medications": [],
        "labs": [
            {"date": "2026-04-22", "test": "eGFR", "value": 55, "unit": "mL/min/1.73m2"},
            {"date": "2026-04-22", "test": "ACR", "value": 90, "unit": "mg/g"},
        ],
    },
    # T2D + HFrEF + atrial fibrillation on warfarin, A1: SGLT2i for HF, NOAC over warfarin, MRA hyperkalemia
    "P008": {
        "demographics": {"age": 58, "sex": "M"},
        "diagnoses": ["Type 2 diabetes", "Heart failure with reduced ejection fraction", "Atrial fibrillation", "Chronic kidney disease"],
        "medications": ["Metformin 1 g twice daily", "Ramipril 5 mg daily", "Bisoprolol 5 mg daily", "Spironolactone 25 mg daily", "Warfarin 5 mg daily"],
        "labs": [
            {"date": "2026-05-05", "test": "eGFR", "value": 41, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-05", "test": "ACR", "value": 25, "unit": "mg/g"},
            {"date": "2026-05-05", "test": "Potassium", "value": 5.2, "unit": "mmol/L"},
        ],
    },
    # Pregnant, CKD A3 on ACEi: drug safety in pregnancy (RASi/SGLT2i/statin would be harmful)
    "P009": {
        "demographics": {"age": 34, "sex": "F"},
        "diagnoses": ["Chronic kidney disease", "Pregnancy"],
        "medications": ["Lisinopril 10 mg daily"],
        "labs": [
            {"date": "2026-06-10", "test": "eGFR", "value": 75, "unit": "mL/min/1.73m2"},
            {"date": "2026-06-10", "test": "ACR", "value": 320, "unit": "mg/g"},
        ],
    },
    # Frail 80-year-old, falls, orthostatic dizziness, OTC NSAID, weight loss: less intensive BP, deprescribing, nutrition
    "P010": {
        "demographics": {"age": 80, "sex": "F"},
        "diagnoses": ["Hypertension", "Chronic kidney disease", "Osteoarthritis"],
        "medications": ["Amlodipine 5 mg daily", "Ramipril 5 mg daily", "Ibuprofen 400 mg as needed"],
        "labs": [
            {"date": "2026-05-12", "test": "eGFR", "value": 33, "unit": "mL/min/1.73m2"},
            {"date": "2026-05-12", "test": "ACR", "value": 45, "unit": "mg/g"},
        ],
    },
    # Gout flare treated with naproxen, CKD G3b A1, on ARB: urate-lowering, acute gout treatment, NSAID harm
    "P011": {
        "demographics": {"age": 50, "sex": "M"},
        "diagnoses": ["Hypertension", "Chronic kidney disease", "Gout"],
        "medications": ["Losartan 50 mg daily", "Naproxen 500 mg twice daily"],
        "labs": [
            {"date": "2026-06-03", "test": "eGFR", "value": 40, "unit": "mL/min/1.73m2"},
            {"date": "2026-06-03", "test": "ACR", "value": 12, "unit": "mg/g"},
            {"date": "2026-06-03", "test": "Uric acid", "value": 10.1, "unit": "mg/dL"},
        ],
    },
    # On hemodialysis: most of this guideline is out of scope; statements restricted to non-KRT must not appear
    "P012": {
        "demographics": {"age": 63, "sex": "M"},
        "diagnoses": ["Type 2 diabetes", "Kidney failure on hemodialysis"],
        "medications": ["Insulin glargine 20 units nightly", "Sevelamer 800 mg with meals"],
        "labs": [
            {"date": "2026-06-15", "test": "eGFR", "value": 8, "unit": "mL/min/1.73m2"},
            {"date": "2026-06-15", "test": "Potassium", "value": 5.3, "unit": "mmol/L"},
        ],
    },
}

# Current-visit notes as a clinician would type them
visit_notes = {
    "P001": "67M, HTN on amlodipine 10. Increasing fatigue over recent months. BP 158/94 today, "
            "still elevated despite meds. Recent bloods: eGFR 48, creat 1.6. No urine ACR on file. What should we do?",
    "P002": "54F known CKD + HTN on ramipril 10mg. Repeat eGFR 38. No new symptoms. BP 138/84. Review management.",
    "P003": "42M routine bloods, eGFR 94, creat 0.9. Asymptomatic, no PMH.",
    "P004": "61F T2D and CKD, on metformin and lisinopril 40 (max dose). eGFR 52, ACR 420, K 4.6, HbA1c 8.2 - "
            "glucose not at target. BP 136/80. Not on SGLT2i. How to optimise kidney protection?",
    "P005": "72M CKD G4, eGFR down from 27 to 22 in 6 months, ACR 850. On losartan 100 + amlodipine. K 5.6. "
            "Reports poor appetite and nausea for 2 months. BP 142/78. Not yet seen by nephrology.",
    "P006": "45F, incidental ACR 150 and microscopic hematuria on dipstick. eGFR 68. No diabetes. BP 142/90 on two readings, "
            "no treatment. No FHx of kidney disease. Uses no contraception, may want another child. Next steps?",
    "P007": "12M with CAKUT and CKD G3a (eGFR 55), ACR 90. Office BP above 95th percentile for age/sex/height on repeat. "
            "Growth tracking along 10th centile. Not on medication. Management plan?",
    "P008": "58M T2D, HFrEF (EF 30%), AF on warfarin (labile INR), CKD eGFR 41, ACR 25. On metformin 1g bd, ramipril 5, "
            "bisoprolol, spironolactone 25. K 5.2. BP 118/70. Review meds.",
    "P009": "34F, 14 weeks pregnant, known CKD, ACR 320, eGFR 75. Still taking lisinopril 10mg. BP 128/82. First visit since conception.",
    "P010": "80F with CKD eGFR 33, ACR 45, HTN on amlodipine + ramipril. BP 150/70 seated, dizzy on standing. "
            "Two falls this year. Takes ibuprofen from the pharmacy for knee pain. Lost 4 kg in 6 months, eats little. Lives alone.",
    "P011": "50M CKD G3b (eGFR 40, ACR 12), HTN on losartan 50. Second gout attack this year (big toe), started naproxen 500 bd. "
            "Urate 10.1. BP 134/84. Should we start urate-lowering?",
    "P012": "63M on hemodialysis 3x/week for 2 years, T2D on insulin. Pre-dialysis K 5.3. No statin. "
            "Asks whether he should start a statin or one of the 'new kidney drugs'.",
}


def state_from_record(record) -> PatientState:
    labs = {}
    for lab in sorted(record["labs"], key=lambda l: l["date"]):
        labs[lab["test"].lower()] = lab
    dx = " ".join(record["diagnoses"]).lower()
    return PatientState(
        age=record["demographics"]["age"],
        sex=record["demographics"]["sex"],
        egfr=labs.get("egfr", {}).get("value"),
        egfr_date=labs.get("egfr", {}).get("date"),
        acr_mg_g=labs.get("acr", {}).get("value"),
        potassium=labs.get("potassium", {}).get("value"),
        known_ckd="chronic kidney disease" in dx or "ckd" in dx,
        hypertension="hypertension" in dx,
        diabetes="type_2" if "type 2 diabetes" in dx else "type_1" if "type 1 diabetes" in dx else "unknown",
        heart_failure="heart failure" in dx or None,
        on_dialysis="dialysis" in dx,
        kidney_transplant="transplant" in dx,
        pregnant=True if "pregnan" in dx else None,
        medications=record["medications"],
    )

In [22]:
NOTE_EXTRACTION_SYSTEM = (
    "Extract patient facts from a clinician's visit note into the schema. "
    "Only record facts explicitly stated. Unknown -> null (or 'unknown' for diabetes). "
    "Convert BP 'a/b' to sbp=a. Do not infer diagnoses from lab values."
)


def state_from_note(note: str) -> PatientState:
    from openai import OpenAI
    client = OpenAI(max_retries=10)
    resp = client.responses.parse(
        model=LLM_MODEL,
        input=[{"role": "system", "content": NOTE_EXTRACTION_SYSTEM}, {"role": "user", "content": note}],
        text_format=PatientState,
    )
    return resp.output_parsed


def state_from_note_offline(note: str) -> PatientState:
    # Minimal regex stand-in so the notebook runs without an API key. Not a replacement for §7 LLM extraction.
    bp = re.search(r"BP\s*(\d{2,3})/(\d{2,3})", note)
    symptoms = [s for s in ["fatigue", "nausea", "itch", "oedema", "edema", "breathless", "weight loss", "poor appetite"] if s in note.lower()]
    return PatientState(sbp=int(bp.group(1)) if bp else None, symptoms=symptoms,
                        concerns=["blood pressure uncontrolled"] if re.search(r"elevated despite|uncontrolled", note, re.I) else [])


def merge_states(record_state: PatientState, note_state: PatientState) -> PatientState:
    merged = record_state.model_dump()
    for key, value in note_state.model_dump().items():
        if key in ("medications", "symptoms", "concerns"):
            merged[key] = sorted(set(merged[key]) | set(value))
        elif key == "diabetes":
            if merged[key] == "unknown":
                merged[key] = value
        elif merged.get(key) is None and value is not None:
            merged[key] = value
    return PatientState(**merged)


def build_patient_state(pid):
    record_state = state_from_record(patients[pid])
    if RUN_LLM:
        note_state = PatientState(**llm_cached(f"note|{visit_notes[pid]}", lambda: state_from_note(visit_notes[pid]).model_dump()))
    else:
        note_state = state_from_note_offline(visit_notes[pid])
    return merge_states(record_state, note_state)


patient_states = {pid: build_patient_state(pid) for pid in patients}
for pid, st in patient_states.items():
    print(pid, st.model_dump(exclude_defaults=True))
    print("   derived:", derive(st))

P001 {'age': 67, 'sex': 'M', 'egfr': 48.0, 'egfr_date': '2026-04-12', 'sbp': 158, 'known_ckd': False, 'hypertension': True, 'diabetes': 'none', 'on_dialysis': False, 'kidney_transplant': False, 'medications': ['Amlodipine 10 mg daily', 'amlodipine'], 'symptoms': ['fatigue'], 'concerns': ['BP remains elevated despite medication', 'Fatigue - cause?', 'No urine ACR on file', 'eGFR lowered (48) - next steps']}
   derived: {'is_adult': True, 'gfr_category': 'G3a', 'acr_category': None, 'risk_category': None, 'drug_classes': ['CCB'], 'missing_data': ['urine ACR']}
P002 {'age': 54, 'sex': 'F', 'egfr': 38.0, 'egfr_date': '2026-03-10', 'sbp': 138, 'known_ckd': True, 'hypertension': True, 'diabetes': 'none', 'on_dialysis': False, 'kidney_transplant': False, 'medications': ['Ramipril 10 mg daily', 'ramipril 10mg'], 'concerns': ['Review management']}
   derived: {'is_adult': True, 'gfr_category': 'G3b', 'acr_category': None, 'risk_category': None, 'drug_classes': ['RASi'], 'missing_data': ['urine 

## 8. Eligibility filter (exclusion only)

- `excluded` — a known patient fact contradicts the statement (child-only for an adult, eGFR outside range…)
- `conditional` — the statement depends on data we don't have (e.g. ACR). **Kept** and flagged:
  that missing data is itself a finding ("SGLT2i may apply — ACR needed").
- `eligible` — nothing contradicts it.

In [23]:
def in_range(value, rng):
    lo, hi = rng
    return lo <= value < hi


def check_applicability(meta, st: PatientState, d: dict):
    reasons, needs = [], []

    if meta["population"] == "child" and d["is_adult"]:
        reasons.append("pediatric-only statement; patient is adult")
    if meta["population"] == "adult" and not d["is_adult"]:
        reasons.append("adult-only statement; patient is a child")

    if meta["egfr_range"]:
        if st.egfr is None:
            needs.append("eGFR")
        elif not in_range(st.egfr, meta["egfr_range"]):
            reasons.append(f"eGFR {st.egfr} outside {meta['egfr_range']}")

    if meta["acr_range"]:
        if st.acr_mg_g is None:
            needs.append("urine ACR")
        elif not in_range(st.acr_mg_g, meta["acr_range"]):
            reasons.append(f"ACR {st.acr_mg_g} outside {meta['acr_range']}")

    if meta["age_range"] and st.age is not None and not in_range(st.age, meta["age_range"]):
        reasons.append(f"age {st.age} outside {meta['age_range']}")

    if meta["diabetes"] == "required":
        if st.diabetes == "none":
            reasons.append("requires diabetes")
        elif st.diabetes == "unknown":
            needs.append("diabetes status")
    if meta["diabetes"] == "excluded" and st.diabetes in ("type_1", "type_2"):
        reasons.append("statement is for people without diabetes")

    on_krt = bool(st.on_dialysis or st.kidney_transplant)
    if meta["excludes_krt"] and on_krt:
        reasons.append("not for dialysis/transplant")
    if meta["requires_krt"] and not on_krt:
        reasons.append("only for dialysis/transplant")

    if meta["mentions_pregnancy"] and st.sex == "M":
        reasons.append("pregnancy-specific")

    status = "excluded" if reasons else "conditional" if needs else "eligible"
    return status, reasons, needs


def filter_statements(st: PatientState):
    # Consensus of regex and LLM metadata: exclude only if both exclude; otherwise keep, with the
    # union of missing data from the opinions that keep it.
    d = derive(st)
    out = {}
    for s in statements:
        opinions = [check_applicability(s["applicability"], st, d)]
        if "applicability_llm" in s:
            opinions.append(check_applicability(s["applicability_llm"], st, d))
        kept = [o for o in opinions if o[0] != "excluded"]
        if not kept:
            reasons = list(dict.fromkeys(r for o in opinions for r in o[1]))
            out[s["statement_id"]] = ("excluded", reasons, [])
        else:
            needs = list(dict.fromkeys(n for o in kept for n in o[2]))
            out[s["statement_id"]] = ("conditional" if needs else "eligible", [], needs)
    return out


applicability_P001 = filter_statements(patient_states["P001"])
print(Counter(v[0] for v in applicability_P001.values()))
pd.DataFrame(
    [(sid, st, "; ".join(r), "; ".join(n)) for sid, (st, r, n) in applicability_P001.items() if st != "eligible"],
    columns=["statement_id", "status", "excluded_because", "needs"],
).head(40)

Counter({'eligible': 135, 'excluded': 28, 'conditional': 6})


,statement_id,status,excluded_because,needs
0,PP 1.2.3.3,excluded,pediatric-only statement; patient is adult,
1,PP 1.2.3.4,excluded,pediatric-only statement; patient is adult,
2,PP 1.2.3.5,excluded,pediatric-only statement; patient is adult,
3,PP 1.2.4.3,excluded,pediatric-only statement; patient is adult,
4,PP 1.3.1.2,conditional,,urine ACR
5,PP 1.3.1.4,excluded,pediatric-only statement; patient is adult,
6,PP 1.4.3,conditional,,urine ACR
7,PP 3.2.2.5,excluded,pediatric-only statement; patient is adult,
8,PP 3.3.1.4,excluded,pediatric-only statement; patient is adult,
9,PP 3.3.2.2,excluded,pediatric-only statement; patient is adult,


## 9. Indexes: BM25 + dense

The indexed text is `statement id | heading path | statement text` — the heading path adds
domain words ("Blood pressure control", "Lipid management") the statement itself may lack.
Rationale is **not** embedded (it would dilute the vector); it is attached after retrieval.

In [24]:
from rank_bm25 import BM25Okapi

STOPWORDS = set("a an the of in and or for to with by on at as is are be be that this these those from use using people person".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:[-/][a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


def index_text(s):
    return f"{s['statement_id']} | {s['heading_path']}\n{s['text']}"


corpus_ids = [s["statement_id"] for s in statements]
corpus_texts = [index_text(s) for s in statements]
stmt_by_id = {s["statement_id"]: s for s in statements}

bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
print("BM25 index:", len(corpus_texts), "docs")

BM25 index: 169 docs


In [25]:
if EMBEDDING_BACKEND == "local":
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer(LOCAL_EMBEDDING_MODEL)
    BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

    def embed_docs(texts):
        return embedder.encode(texts, normalize_embeddings=True, batch_size=32)

    def embed_query(text):
        return embedder.encode([BGE_QUERY_PREFIX + text], normalize_embeddings=True)[0]
else:
    from openai import OpenAI
    _oai = OpenAI(max_retries=10)

    def _embed(texts):
        resp = _oai.embeddings.create(model=OPENAI_EMBEDDING_MODEL, input=texts)
        v = np.array([d.embedding for d in resp.data])
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    embed_docs = _embed

    def embed_query(text):
        return _embed([text])[0]

doc_vectors = embed_docs(corpus_texts)
print("dense index:", doc_vectors.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dense index: (169, 768)


## 10. Retrieval strategies

We compare, from naive to full:

| name | query | candidates | ranking |
|---|---|---|---|
| `dense_raw` | raw patient text (original approach) | all | cosine |
| `hybrid_raw` | raw patient text | all | BM25 + dense, RRF |
| `filtered_hybrid_raw` | raw patient text | eligible + conditional | BM25 + dense, RRF |
| `filtered_multiquery` | care-domain sub-queries | eligible + conditional | per sub-query RRF, then fused |
| `filtered_multiquery_rerank` | care-domain sub-queries | eligible + conditional | + cross-encoder per sub-query |

In [26]:
def rank_dense(query, candidate_ids):
    q = embed_query(query)
    idx = [corpus_ids.index(c) for c in candidate_ids]
    scores = doc_vectors[idx] @ q
    order = np.argsort(-scores)
    return [(candidate_ids[i], float(scores[i])) for i in order]


def rank_bm25(query, candidate_ids):
    scores = bm25.get_scores(tokenize(query))
    pairs = [(c, float(scores[corpus_ids.index(c)])) for c in candidate_ids]
    return sorted(pairs, key=lambda x: -x[1])


def rrf(rankings, k=60):
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, (doc_id, _) in enumerate(ranking):
            fused[doc_id] += 1.0 / (k + rank + 1)
    return sorted(fused.items(), key=lambda x: -x[1])


def hybrid(query, candidate_ids):
    return rrf([rank_dense(query, candidate_ids), rank_bm25(query, candidate_ids)])

### Query decomposition: care-domain sub-queries

For a "what's missing in this patient's care" task, a **fixed checklist of care domains** driven
by the patient state is more reliable than asking an LLM to free-form decompose. Each sub-query
is phrased like guideline text, with the patient's facts filled in. The LLM variant is
provided for comparison (gated).

In [27]:
def build_subqueries(st: PatientState) -> list[str]:
    d = derive(st)
    g = d["gfr_category"] or "unknown GFR category"
    pop = "adults" if d["is_adult"] else "children"
    q = []

    if st.egfr is not None and st.egfr < 60 and not st.known_ckd:
        q.append(f"confirm chronicity of CKD after a single low eGFR; repeat testing; evaluate cause of CKD in {pop}")
    q.append(f"test urine albumin-to-creatinine ratio ACR and eGFR to detect and stage CKD {g} in {pop}")
    if st.egfr is not None and st.egfr < 60:
        q += [
            f"frequency of monitoring eGFR and albuminuria in people with CKD {g}",
            f"kidney failure risk prediction equation and criteria for referral to nephrology in CKD {g}",
            f"statin treatment to reduce cardiovascular risk in {pop} aged {st.age} with eGFR {st.egfr:g}",
            f"SGLT2 inhibitor treatment for {pop} with CKD eGFR {st.egfr:g}",
            f"renin-angiotensin system inhibitor ACEi or ARB for CKD with albuminuria",
            f"medication management, drug dosing and nephrotoxic drug avoidance with reduced eGFR {st.egfr:g}",
        ]
    if st.hypertension or (st.sbp and st.sbp >= 130):
        q.append(f"blood pressure target and antihypertensive treatment for {pop} with CKD and high blood pressure")
    if "RASi" in d["drug_classes"]:
        q.append("monitoring potassium and creatinine in people on ACEi or ARB; continue RASi when eGFR declines")
    if st.diabetes in ("type_1", "type_2"):
        q.append("glucose-lowering therapy SGLT2i GLP-1 receptor agonist and finerenone in CKD with type 2 diabetes")
    for symptom in st.symptoms:
        q.append(f"assessment and management of {symptom} and uremic symptoms in CKD")
    if st.potassium and st.potassium > 5.0:
        q.append("management of hyperkalemia in CKD")
    return q


class SubQueries(BaseModel):
    queries: list[str] = Field(description="4-10 short retrieval queries, each about one clinical question, phrased like guideline text")


def build_subqueries_llm(st: PatientState, note: str) -> list[str]:
    from openai import OpenAI
    client = OpenAI(max_retries=10)
    resp = client.responses.parse(
        model=LLM_MODEL,
        input=[
            {"role": "system", "content": "Decompose the clinical situation into separate guideline look-up questions "
                                          "(diagnosis/staging, monitoring, each relevant treatment class, referral, symptoms). "
                                          "One question per item. Include key patient facts (age, eGFR, conditions) in each."},
            {"role": "user", "content": f"Patient state: {st.model_dump_json()}\nDerived: {derive(st)}\nNote: {note}"},
        ],
        text_format=SubQueries,
    )
    return resp.output_parsed.queries


for q in build_subqueries(patient_states["P001"]):
    print("-", q)

- confirm chronicity of CKD after a single low eGFR; repeat testing; evaluate cause of CKD in adults
- test urine albumin-to-creatinine ratio ACR and eGFR to detect and stage CKD G3a in adults
- frequency of monitoring eGFR and albuminuria in people with CKD G3a
- kidney failure risk prediction equation and criteria for referral to nephrology in CKD G3a
- statin treatment to reduce cardiovascular risk in adults aged 67 with eGFR 48
- SGLT2 inhibitor treatment for adults with CKD eGFR 48
- renin-angiotensin system inhibitor ACEi or ARB for CKD with albuminuria
- medication management, drug dosing and nephrotoxic drug avoidance with reduced eGFR 48
- blood pressure target and antihypertensive treatment for adults with CKD and high blood pressure
- assessment and management of fatigue and uremic symptoms in CKD


In [28]:
from sentence_transformers import CrossEncoder

_rerankers = {}


def get_reranker(name):
    if name not in _rerankers:
        _rerankers[name] = CrossEncoder(RERANKERS[name])
    return _rerankers[name]


def rerank(query, doc_ids, reranker_name):
    ce = get_reranker(reranker_name)
    scores = ce.predict([(query, index_text(stmt_by_id[i])) for i in doc_ids])
    return sorted(zip(doc_ids, map(float, scores)), key=lambda x: -x[1])


def interleave(per_query_rankings, limit):
    # round-robin across sub-queries so every care domain gets represented
    out, seen = [], set()
    for rank in range(max(len(r) for r in per_query_rankings)):
        for ranking in per_query_rankings:
            if rank < len(ranking) and ranking[rank][0] not in seen:
                seen.add(ranking[rank][0])
                out.append(ranking[rank])
    return out[:limit]


def raw_query_text(pid):
    st = patient_states[pid]
    return (f"{st.age}-year-old patient. " + " ".join(patients[pid]["diagnoses"]) +
            f". eGFR {st.egfr}. " + visit_notes[pid] + " What CKD guideline recommendations may be relevant?")


def retrieve(pid, strategy, k=15, reranker_name="bge-reranker-v2-m3", per_query=8, subqueries=None):
    st = patient_states[pid]
    applicability = filter_statements(st)
    allowed = [sid for sid in corpus_ids if applicability[sid][0] != "excluded"]

    if strategy == "dense_raw":
        return rank_dense(raw_query_text(pid), corpus_ids)[:k]
    if strategy == "hybrid_raw":
        return hybrid(raw_query_text(pid), corpus_ids)[:k]
    if strategy == "filtered_hybrid_raw":
        return hybrid(raw_query_text(pid), allowed)[:k]

    if strategy == "llm_screen":
        return [(sid, 0.0) for sid in llm_screen(pid)][:k]
    if strategy == "filtered_llmquery_rerank":
        subqueries = llm_cached(f"subqueries|{pid}|{st.model_dump_json()}|{visit_notes[pid]}",
                                lambda: build_subqueries_llm(st, visit_notes[pid]))
        strategy = "filtered_multiquery_rerank"

    queries = subqueries or build_subqueries(st)
    candidates = [hybrid(q, allowed)[: per_query * 2] for q in queries]
    if strategy == "filtered_multiquery":
        return interleave([r[:per_query] for r in candidates], k)
    if strategy == "filtered_multiquery_rerank":
        reranked = [rerank(q, [i for i, _ in r], reranker_name)[:per_query] for q, r in zip(queries, candidates)]
        return interleave(reranked, k)
    raise ValueError(strategy)

class Screening(BaseModel):
    relevant: list[str] = Field(description="statement_ids relevant to this visit, most important first")


def llm_screen(pid):
    # Baseline: after filtering, give the LLM every remaining statement (~8k tokens) and let it pick.
    from openai import OpenAI
    st = patient_states[pid]
    applicability = filter_statements(st)
    listing = "\n".join(
        f"[{sid}] ({stmt_by_id[sid]['domain']}) {stmt_by_id[sid]['text']}"
        + (f"  <needs: {', '.join(applicability[sid][2])}>" if applicability[sid][2] else "")
        for sid in corpus_ids if applicability[sid][0] != "excluded"
    )
    prompt = f"PATIENT\n{st.model_dump_json()}\nDerived: {derive(st)}\nNote: {visit_notes[pid]}\n\nSTATEMENTS\n{listing}"

    def call():
        resp = OpenAI(max_retries=10).responses.parse(
            model=LLM_MODEL,
            input=[
                {"role": "system", "content": "Select the guideline statements a clinician should review for this visit, "
                                              "most important first (at most 25). Return only statement_ids from the list."},
                {"role": "user", "content": prompt},
            ],
            text_format=Screening,
        )
        return resp.output_parsed.relevant

    return [sid for sid in llm_cached(f"screen|{prompt}", call) if sid in stmt_by_id]


## 11. Evaluation

**You can't tune retrieval without a gold set.** Below is a DRAFT gold set (written by Claude, reading all 169
statements against each synthetic patient). ⚠️ It must be reviewed by a clinician — treat
the numbers as relative comparisons between strategies, not absolute quality.

- `must` = a clinician would expect to see it for this visit
- `inappropriate` = retrieving it signals an applicability failure (wrong population / threshold)

Metrics: recall@k on `must`, and the count of `inappropriate` statements in the top k.

In [29]:
# Hand-written lists (not derived from the applicability metadata, so the filter is actually tested)
PEDIATRIC_ONLY = [
    "PP 1.2.3.3", "PP 1.2.3.4", "PP 1.2.3.5", "PP 1.2.4.3", "PP 1.3.1.4", "PP 3.2.2.5", "PP 3.3.1.4", "PP 3.3.2.2",
    "Rec 3.4.2", "PP 3.4.2", "PP 3.4.3", "PP 5.1.2", "PP 5.4.5", "PP 5.3.1.1.1", "PP 5.3.1.1.2", "PP 5.3.1.1.3",
]
YOUNG_PEOPLE_ONLY = ["PP 5.3.1.2.1", "PP 5.3.1.2.2", "PP 5.3.1.2.3"]  # under 25 / after transfer from pediatric care
ADULT_ONLY = [
    "Rec 1.1.2.1", "Rec 3.3.1.1", "PP 3.3.1.1", "PP 3.3.1.2", "PP 3.3.1.5", "Rec 3.4.1", "Rec 3.7.2", "Rec 3.7.3",
    "Rec 3.8.1", "Rec 3.9.1", "Rec 3.15.1.1", "Rec 3.15.1.2", "Rec 3.15.1.3", "PP 3.15.1.3", "PP 5.1.1", "PP 5.4.3",
]
NOT_FOR_ADULTS = PEDIATRIC_ONLY + YOUNG_PEOPLE_ONLY

GOLD = {
    "P001": {  # 67M, HTN on amlodipine, eGFR 48 (single value), no ACR, fatigue, BP uncontrolled
        "must": [
            "PP 1.1.1.1",    # test ACR + GFR
            "PP 1.1.1.2",    # repeat to confirm after incidental low eGFR
            "PP 1.1.3.2",    # don't assume chronicity from one value
            "PP 1.1.4.1",    # establish cause
            "Rec 1.1.2.1",   # eGFRcr / cystatin C for staging
            "PP 2.1.1",      # monitor ACR + GFR at least annually
            "Rec 3.4.1",     # SBP target <120
            "Rec 3.15.1.1",  # statin: age >=50, eGFR <60
            "Rec 3.7.2",     # SGLT2i — conditional on ACR
            "Rec 3.6.2",     # RASi for A2 without diabetes — conditional on ACR
            "Rec 2.2.1",     # kidney failure risk equation in G3-G5 (needs ACR)
            "PP 5.2.2.1",    # ask about uremic symptoms (fatigue)
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.15.1.2", "Rec 3.15.1.3"],
    },
    "P002": {  # 54F, known CKD + HTN on ramipril, eGFR 38, no ACR
        "must": [
            "PP 1.1.1.1",    # ACR + GFR testing
            "PP 2.1.1",      # monitor at least annually
            "Rec 2.2.1",     # kidney failure risk equation in G3-G5
            "Rec 3.4.1",     # SBP target <120
            "Rec 3.15.1.1",  # statin: age >=50, eGFR <60
            "Rec 3.7.2",     # SGLT2i — conditional on ACR
            "PP 3.6.1",      # RASi at highest tolerated dose
            "PP 5.1.1",      # referral criteria (Figure 48)
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.15.1.2", "Rec 3.15.1.3"],
    },
    "P003": {  # 42M, eGFR 94, no risk factors — healthy control: only checks nothing inappropriate is surfaced
        "must": [],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 2.2.1", "Rec 3.3.1.1", "Rec 3.15.1.1", "Rec 3.15.1.2", "PP 3.16.2", "PP 4.4.2.1", "PP 5.4.3"],
    },
    "P004": {  # 61F T2D, eGFR 52, ACR 420 on max-dose lisinopril + metformin, HbA1c 8.2
        "must": [
            "PP 2.1.1",      # monitor ACR + GFR
            "Rec 2.2.1",     # kidney failure risk equation (G3a)
            "Rec 3.4.1",     # SBP target <120 (BP 136)
            "Rec 3.6.3",     # RASi with diabetes and A2-A3
            "PP 3.6.1",      # RASi at maximal tolerated dose
            "Rec 3.7.1",     # SGLT2i: T2D, eGFR >=20
            "Rec 3.7.2",     # SGLT2i: ACR >=200
            "Rec 3.8.1",     # nonsteroidal MRA: T2D, eGFR >25, normal K, albuminuria despite max RASi
            "PP 3.8.2",      # MRA added to RASi + SGLT2i
            "PP 3.8.3",      # MRA potassium monitoring
            "Rec 3.9.1",     # GLP-1 RA: glycemic target not met on metformin + SGLT2i
            "Rec 3.15.1.1",  # statin: >=50, eGFR <60
            "PP 5.1.1",      # referral: ACR >=300
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.6.1", "Rec 3.6.2", "Rec 3.7.3", "Rec 3.15.1.2", "Rec 3.15.1.3"],
    },
    "P005": {  # 72M CKD G4 A3, rapid decline, K 5.6 on losartan, poor appetite + nausea
        "must": [
            "Rec 2.2.1",     # kidney failure risk equation
            "PP 2.2.2",      # 2-year risk >10% -> multidisciplinary care
            "PP 2.2.3",      # 2-year risk >40% -> KRT preparation
            "PP 5.1.1",      # refer to nephrology
            "PP 5.4.3",      # plan preemptive transplant / dialysis access at GFR <15-20
            "PP 5.5.1",      # inform about KRT options and conservative care
            "PP 5.3.1",      # multidisciplinary care team
            "PP 3.6.3",      # manage hyperkalemia rather than stopping RASi
            "PP 3.6.7",      # continue RASi below eGFR 30
            "PP 3.11.5.1",   # individualized hyperkalemia management, G3-G5
            "PP 3.11.5.2",   # limit bioavailable potassium foods
            "PP 5.2.2.1",    # ask about uremic symptoms
            "PP 5.2.3.2",    # malnutrition screening (G4, >65, poor appetite)
            "Rec 3.7.2",     # SGLT2i: eGFR >=20, ACR >=200
            "Rec 3.15.1.1",  # statin
            "Rec 3.4.1",     # SBP target (BP 142)
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.7.1", "Rec 3.7.3", "Rec 3.8.1", "Rec 3.15.1.2", "Rec 3.15.1.3", "Rec 3.6.3"],
    },
    "P006": {  # 45F, new ACR 150 + hematuria, eGFR 68, no diabetes, untreated BP 142/90, may want pregnancy
        "must": [
            "PP 1.1.1.2",    # repeat tests to confirm
            "PP 1.3.1.2",    # confirm albuminuria with lab ACR / first morning sample
            "PP 1.1.3.2",    # don't assume chronicity from one value
            "PP 1.1.4.1",    # establish cause (hematuria + albuminuria)
            "Rec 1.1.4.1",   # kidney biopsy to evaluate cause
            "Rec 3.4.1",     # SBP target <120
            "Rec 3.6.2",     # RASi for A2 without diabetes
            "PP 3.6.2",      # check BP, creatinine, K 2-4 weeks after starting RASi
            "PP 4.1.4",      # child-bearing potential: teratogenicity counseling (ACEi/ARB)
            "Rec 3.3.2.1",   # sodium <2 g/day
            "PP 2.1.1",      # monitor ACR + GFR
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.6.3", "Rec 3.3.1.1", "Rec 2.2.1", "Rec 3.7.3", "Rec 3.15.1.1", "Rec 3.15.1.2", "PP 5.4.3"],
    },
    "P007": {  # 12M, CAKUT, CKD G3a A2, BP >95th percentile
        "must": [
            "Rec 3.4.2",     # pediatric BP target (24-h MAP by ABPM)
            "PP 3.4.2",      # ABPM yearly, office BP every 3-6 months
            "PP 3.4.3",      # office SBP target 50th-75th percentile
            "PP 3.3.1.4",    # don't restrict protein in children
            "PP 3.3.2.2",    # sodium intake counseling for children with high BP
            "PP 3.2.2.5",    # physical activity for children
            "PP 1.3.1.4",    # children: first morning urine PCR + ACR
            "PP 1.2.4.3",    # pediatric eGFR equations
            "PP 2.1.1",      # albuminuria/proteinuria + GFR annually (children)
            "PP 5.1.2",      # pediatric referral criteria
            "PP 5.3.1.1.1",  # prepare for transfer to adult care from 11-14 years
        ],
        "inappropriate": ADULT_ONLY,
    },
    "P008": {  # 58M T2D, HFrEF, AF on warfarin, eGFR 41, ACR 25, on spironolactone, K 5.2
        "must": [
            "Rec 3.7.1",     # SGLT2i: T2D, eGFR >=20
            "Rec 3.7.2",     # SGLT2i: heart failure irrespective of albuminuria
            "Rec 3.16.1",    # NOAC in preference to warfarin, G1-G4
            "PP 3.16.2",     # NOAC dose adjustment for GFR
            "PP 3.6.6",      # RASi in A1 for heart failure
            "PP 3.8.5",      # steroidal MRA: hyperkalemia / GFR decline risk
            "PP 3.6.3",      # manage hyperkalemia, keep RASi
            "PP 4.2.1",      # dose renally cleared drugs by GFR (metformin)
            "PP 4.3.1",      # medication review
            "Rec 3.15.1.1",  # statin: >=50, eGFR <60
            "Rec 2.2.1",     # kidney failure risk equation
            "PP 2.1.1",      # monitor ACR + GFR
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.8.1", "Rec 3.6.1", "Rec 3.6.2", "Rec 3.6.3", "Rec 3.15.1.2", "Rec 3.15.1.3"],
    },
    "P009": {  # 34F, 14 weeks pregnant, CKD A3, still on lisinopril
        "must": [
            "PP 4.1.4",      # teratogenicity review for people of child-bearing potential
            "PP 4.1.1",      # benefits vs harms of medications in CKD
            "PP 4.3.1",      # medication review at transition of care
            "PP 5.1.1",      # referral (ACR >=300)
            "PP 2.1.2",      # monitor more often when at higher risk
            "PP 1.1.4.1",    # establish cause of CKD
        ],
        # starting / maximizing drugs that are contraindicated in pregnancy
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.6.1", "PP 3.6.1", "Rec 3.7.2", "Rec 3.7.3", "Rec 3.15.1.3", "Rec 3.15.1.1", "Rec 3.15.1.2"],
    },
    "P010": {  # 80F frail, falls, orthostatic dizziness, OTC ibuprofen, weight loss, eGFR 33, ACR 45
        "must": [
            "PP 3.4.1",      # less intensive BP lowering in frailty / falls / postural hypotension
            "PP 4.1.3",      # review OTC medicines (ibuprofen)
            "PP 4.1.1",      # nephrotoxic medications: benefit vs harm
            "PP 4.3.1",      # medication review
            "PP 5.2.3.2",    # malnutrition screening (>65, weight loss, poor appetite)
            "PP 5.2.3.3",    # medical nutrition therapy
            "PP 3.3.1.5",    # older adults with frailty: higher protein/calorie targets
            "PP 3.2.2.3",    # falls risk: physical activity advice
            "Rec 3.7.3",     # SGLT2i: eGFR 20-45, ACR <200
            "PP 3.6.1",      # RASi dosing (already on ramipril, A2)
            "Rec 3.15.1.1",  # statin
            "Rec 2.2.1",     # kidney failure risk equation
            "PP 2.3.2",      # mortality risk prediction for goals of care
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.15.1.2", "Rec 3.15.1.3", "PP 3.3.1.2", "Rec 3.6.3", "Rec 3.7.1"],
    },
    "P011": {  # 50M CKD G3b A1, recurrent gout on naproxen, urate 10.1, on losartan
        "must": [
            "Rec 3.14.1",    # symptomatic hyperuricemia -> urate-lowering
            "PP 3.14.1",     # consider ULT after first gout episode
            "PP 3.14.2",     # xanthine oxidase inhibitors preferred
            "PP 3.14.3",     # acute gout: colchicine/glucocorticoids rather than NSAIDs
            "PP 3.14.4",     # nonpharmacological gout prevention
            "PP 4.1.1",      # nephrotoxic medications (NSAID)
            "Rec 3.7.3",     # SGLT2i: eGFR 20-45, ACR <200
            "Rec 3.15.1.1",  # statin: >=50, eGFR <60
            "Rec 2.2.1",     # kidney failure risk equation
            "Rec 3.4.1",     # SBP target (134)
        ],
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.6.1", "Rec 3.6.2", "Rec 3.6.3", "Rec 3.7.1", "Rec 3.8.1", "Rec 3.15.1.2", "Rec 3.15.1.3"],
    },
    "P012": {  # 63M on hemodialysis, T2D, asks about statin / new kidney drugs
        "must": [
            "PP 4.2.1",      # dose medications by GFR
            "PP 4.3.1",      # medication review
        ],
        # statin/SGLT2i/MRA initiation recommendations are for CKD not on dialysis
        "inappropriate": NOT_FOR_ADULTS + ["Rec 3.15.1.1", "Rec 3.15.1.3", "Rec 3.7.1", "Rec 3.7.2", "Rec 3.7.3", "Rec 3.8.1", "Rec 3.3.1.1", "PP 3.3.1.2", "PP 5.4.3"],
    },
}


### Clinician review of the gold set (spreadsheet round-trip)

For each patient, this writes `outputs/gold_review_<patient>.csv` with **all 169 statements**,
sorted by care domain, showing what the filter decided (`eligible` / `conditional` / `excluded`
and why) and the draft label.

The clinician fills `your_label` for each row that matters (leave the rest blank):

| label | meaning |
|---|---|
| `must` | I would expect to see this for this visit — missing it is a retrieval failure |
| `should` | useful, but not essential |
| `inappropriate` | showing this for this patient would be wrong or misleading |

Save as `outputs/gold_reviewed_<patient>.csv`. When it exists it **replaces** the draft for that
patient. A `must` on a row the filter `excluded` is reported below as a **filter error**.

In [30]:
def export_gold_template(pid):
    st = patient_states[pid]
    applicability = filter_statements(st)
    draft = GOLD.get(pid, {"must": [], "inappropriate": []})
    rows = []
    for s in sorted(statements, key=lambda x: (x["domain"] or "", corpus_ids.index(x["statement_id"]))):
        sid = s["statement_id"]
        status, reasons, needs = applicability[sid]
        rows.append({
            "statement_id": sid,
            "domain": s["domain"],
            "text": s["text"],
            "filter_status": status,
            "excluded_because": "; ".join(reasons),
            "needs_data": "; ".join(needs),
            "draft_label": "must" if sid in draft["must"] else "inappropriate" if sid in draft["inappropriate"] else "",
            "your_label": "",
            "notes": "",
        })
    path = OUTPUT_DIR / f"gold_review_{pid}.csv"
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")
    return path


def load_gold_review(pid):
    path = OUTPUT_DIR / f"gold_reviewed_{pid}.csv"
    if not path.exists():
        return None
    df = read_review_csv(path)
    label = df["your_label"].str.strip().str.lower()
    unknown_labels = set(label) - {"", "must", "should", "inappropriate"}
    if unknown_labels:
        print(f"{pid}: ignoring unknown labels {unknown_labels}")
    return {
        "must": df.loc[label == "must", "statement_id"].tolist(),
        "should": df.loc[label == "should", "statement_id"].tolist(),
        "inappropriate": df.loc[label == "inappropriate", "statement_id"].tolist(),
        "source": "clinician",
    }


for pid in patients:
    print("wrote", export_gold_template(pid))
    reviewed = load_gold_review(pid)
    if reviewed:
        GOLD[pid] = reviewed
        print(f"  {pid}: using clinician gold set ({len(reviewed['must'])} must, {len(reviewed['inappropriate'])} inappropriate)")

# Guard against typos in the gold set
for pid, g in GOLD.items():
    unknown = [sid for sid in g["must"] + g["inappropriate"] if sid not in stmt_by_id]
    assert not unknown, (pid, unknown)

# Filter errors: statements the clinician says are essential but the filter removed
filter_errors = []
for pid, g in GOLD.items():
    applicability = filter_statements(patient_states[pid])
    for sid in g["must"]:
        status, reasons, _ = applicability[sid]
        if status == "excluded":
            filter_errors.append({"patient": pid, "statement_id": sid, "excluded_because": "; ".join(reasons)})
print("filter errors:", len(filter_errors))
pd.DataFrame(filter_errors)

wrote outputs/gold_review_P001.csv
wrote outputs/gold_review_P002.csv
wrote outputs/gold_review_P003.csv
wrote outputs/gold_review_P004.csv
wrote outputs/gold_review_P005.csv
wrote outputs/gold_review_P006.csv
wrote outputs/gold_review_P007.csv
wrote outputs/gold_review_P008.csv
wrote outputs/gold_review_P009.csv
wrote outputs/gold_review_P010.csv
wrote outputs/gold_review_P011.csv
wrote outputs/gold_review_P012.csv
filter errors: 0


""


In [31]:
def evaluate(strategies, ks=(10, 15, 25), **kwargs):
    rows = []
    for pid, gold in GOLD.items():
        for strategy in strategies:
            ranked = [sid for sid, _ in retrieve(pid, strategy, k=max(ks), **kwargs)]
            row = {"patient": pid, "strategy": strategy}
            for k in ks:
                top = ranked[:k]
                # patients with no "must" (e.g. a healthy control) only contribute inappropriate@k
                row[f"recall@{k}"] = round(sum(s in top for s in gold["must"]) / len(gold["must"]), 2) if gold["must"] else None
                row[f"inappropriate@{k}"] = sum(s in top for s in gold["inappropriate"])
            rows.append(row)
    return pd.DataFrame(rows)


STRATEGIES = ["dense_raw", "hybrid_raw", "filtered_hybrid_raw", "filtered_multiquery", "filtered_multiquery_rerank"]
if RUN_LLM:
    STRATEGIES += ["filtered_llmquery_rerank", "llm_screen"]
results_df = evaluate(STRATEGIES)
results_df

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

,patient,strategy,recall@10,inappropriate@10,recall@15,inappropriate@15,recall@25,inappropriate@25
0,P001,dense_raw,0.25,1,0.50,1,0.58,2
1,P001,hybrid_raw,0.25,1,0.42,2,0.58,2
2,P001,filtered_hybrid_raw,0.33,0,0.58,0,0.58,0
3,P001,filtered_multiquery,0.58,0,0.75,0,0.75,0
4,P001,filtered_multiquery_rerank,0.50,0,0.67,0,0.75,0
...,...,...,...,...,...,...,...,...
79,P012,filtered_hybrid_raw,0.00,1,0.00,1,0.00,2
80,P012,filtered_multiquery,0.00,0,0.00,0,0.00,0
81,P012,filtered_multiquery_rerank,0.00,1,0.00,1,0.00,1
82,P012,filtered_llmquery_rerank,0.00,0,0.00,0,0.00,2


In [32]:
results_df.groupby("strategy", sort=False)[[c for c in results_df.columns if "@" in c]].mean().round(2)

,recall@10,inappropriate@10,recall@15,inappropriate@15,recall@25,inappropriate@25
strategy,,,,,,
dense_raw,0.27,2.75,0.33,3.67,0.39,4.92
hybrid_raw,0.27,2.42,0.34,3.67,0.40,5.17
filtered_hybrid_raw,0.30,0.42,0.37,0.67,0.46,0.75
filtered_multiquery,0.24,0.25,0.34,0.33,0.40,0.42
filtered_multiquery_rerank,0.26,0.33,0.38,0.33,0.44,0.50
filtered_llmquery_rerank,0.37,0.00,0.46,0.25,0.52,0.58
llm_screen,0.44,0.17,0.64,0.17,0.73,0.17


### Findings (draft gold set, 12 synthetic patients, 2026-09-25)

- **Filter:** 0 filter errors with regex+LLM consensus. Either source alone produced 5–9 (regex: "A or B"
  read as "A required"; LLM: invented "diabetes required", wrong G-category ranges).
- **Best strategy: `llm_screen`** (filter, then the LLM picks from all remaining statements):
  recall@15 ≈ 0.64, recall@25 ≈ 0.73, ~0.2 inappropriate in top 15. Embedding retrieval tops out at ≈ 0.46@15.
- **What it still misses:** routine "background" care the visit note doesn't ask about (statin, kidney failure
  risk equation, annual ACR/GFR monitoring), and Rec 3.16.1 (NOAC over warfarin) for P008.
- **Known safety gap:** pregnancy (P009) — the filter has no rule for drugs contraindicated in pregnancy, so
  RASi statements (Rec 3.6.1, PP 3.6.1) are still shown. The generation step must handle this, or add a rule.
- Numbers are only as good as the draft gold labels — clinician review of `gold_review_*.csv` pending.


In [33]:
# Reranker comparison on the full pipeline
pd.concat([
    evaluate(["filtered_multiquery_rerank"], reranker_name=name).assign(reranker=name)
    for name in RERANKERS
]).groupby("reranker")[["recall@10", "recall@15", "recall@25"]].mean().round(2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,recall@10,recall@15,recall@25
reranker,,,
bge-reranker-v2-m3,0.26,0.38,0.44
medcpt,0.25,0.35,0.40


In [34]:
def show_retrieval(pid, strategy, k=15, **kwargs):
    st = patient_states[pid]
    applicability = filter_statements(st)
    must = set(GOLD.get(pid, {}).get("must", []))
    rows = []
    for sid, score in retrieve(pid, strategy, k=k, **kwargs):
        status, _, needs = applicability[sid]
        rows.append({"statement_id": sid, "gold": "✓" if sid in must else "",
                     "status": status, "needs": ", ".join(needs), "score": round(score, 3),
                     "domain": stmt_by_id[sid]["domain"], "text": stmt_by_id[sid]["text"][:110]})
    df = pd.DataFrame(rows)
    missed = must - set(df["statement_id"])
    print(f"{pid} / {strategy}: missed gold -> {sorted(missed)}")
    return df


show_retrieval("P001", "dense_raw")

P001 / dense_raw: missed gold -> ['PP 1.1.1.1', 'PP 1.1.4.1', 'PP 2.1.1', 'PP 5.2.2.1', 'Rec 2.2.1', 'Rec 3.4.1']


,statement_id,gold,status,needs,score,domain,text
0,Rec 3.7.3,,conditional,urine ACR,0.742,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),Recommendation 3.7.3: We suggest treating adults with eGFR 20 to 45 ml/min per 1.73 m2 with urine ACR <200 mg/
1,PP 3.6.6,,conditional,urine ACR,0.733,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.6: Consider starting people with CKD with normal to mildly increased albuminuria (A1) on RA
2,PP 3.6.5,,eligible,,0.729,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.5: Consider reducing the dose or discontinuing ACEi or ARB in the setting of either symptom
3,PP 1.1.1.2,✓,eligible,,0.728,1.1 Detection and evaluation of CKD,"Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR),"
4,PP 2.1.3,,eligible,,0.724,2.1 Overview on monitoring for progression of CKD based upon GFR and ACR categories,"Practice Point 2.1.3: For people with CKD, a change in eGFR of >20% on a subsequent test exceeds the expected"
5,Rec 3.7.2,✓,eligible,,0.721,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),Recommendation 3.7.2: We recommend treating adults with CKD with an SGLT2i for the following (1A):\n eGFR ≥20
6,Rec 3.15.1.1,✓,eligible,,0.720,3.15 Cardiovascular disease (CVD) and additional specific interventions to modify risk,Recommendation 3.15.1.1: In adults aged ≥50 years with eGFR <60 ml/min per 1.73 m2 but not treated with chroni
7,Rec 3.15.1.2,,excluded,,0.717,3.15 Cardiovascular disease (CVD) and additional specific interventions to modify risk,Recommendation 3.15.1.2: In adults aged ≥50 years with CKD and eGFR ≥60 ml/min per 1.73 m2 (GFR categories G1-
8,PP 3.6.2,,eligible,,0.716,3.6 Renin-angiotensin system inhibitors,"Practice Point 3.6.2: Changes in BP, serum creatinine, and serum potassium should be checked within 2-4 weeks"
9,PP 1.2.2.8,,eligible,,0.713,1.2 Evaluation of GFR,Practice Point 1.2.2.8: Consider timed urine collections for measured creatinine clearance if mGFR is not avai


In [35]:
show_retrieval("P001", "filtered_multiquery_rerank")

P001 / filtered_multiquery_rerank: missed gold -> ['PP 1.1.1.1', 'PP 1.1.3.2', 'PP 1.1.4.1', 'Rec 3.6.2']


,statement_id,gold,status,needs,score,domain,text
0,PP 1.1.1.2,✓,eligible,,0.987,1.1 Detection and evaluation of CKD,"Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR),"
1,PP 2.1.1,✓,eligible,,0.886,2.1 Overview on monitoring for progression of CKD based upon GFR and ACR categories,"Practice Point 2.1.1: Assess albuminuria in adults, or albuminuria/proteinuria in children, and GFR at least a"
2,PP 2.2.1,,eligible,,0.968,2.2 Risk prediction in people with CKD,Practice Point 2.2.1: A 5-year kidney failure risk of 3%-5% can be used to determine need for nephrology refer
3,Rec 3.15.1.1,✓,eligible,,0.191,3.15 Cardiovascular disease (CVD) and additional specific interventions to modify risk,Recommendation 3.15.1.1: In adults aged ≥50 years with eGFR <60 ml/min per 1.73 m2 but not treated with chroni
4,Rec 3.7.2,✓,eligible,,0.952,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),Recommendation 3.7.2: We recommend treating adults with CKD with an SGLT2i for the following (1A):\n eGFR ≥20
5,Rec 3.6.1,,conditional,urine ACR,0.999,3.6 Renin-angiotensin system inhibitors,Recommendation 3.6.1: We recommend starting renin-angiotensin-system inhibitors (RASi) (angiotensin-converting
6,PP 4.1.2,,eligible,,0.571,4.1 Medication choices and monitoring for safety,"Practice Point 4.1.2: Monitor eGFR, electrolytes, and therapeutic medication levels, when indicated, in people"
7,Rec 3.4.1,✓,eligible,,0.992,3.4 Blood pressure control,Recommendation 3.4.1: We suggest that adults with high BP and CKD be treated with a target systolic blood pres
8,PP 5.2.2.1,✓,eligible,,0.977,5.2 Symptoms in CKD,"Practice Point 5.2.2.1: Ask people with progressive CKD about uremic symptoms (e.g., reduced appetite, nausea,"
9,PP 1.1.3.1,,eligible,,0.904,1.1 Detection and evaluation of CKD,Practice Point 1.1.3.1: Proof of chronicity (duration of a minimum of 3 months) can be established by:\n(i) rev


## 12. LLM screening baseline

With only ~170 statements (≈8k tokens), after filtering we can simply give the LLM **every**
eligible statement and ask it to select the relevant ones (`llm_screen`, defined with `retrieve`
in §10 and scored in the §11 table above). Per-patient view:


In [36]:
if RUN_LLM:
    for pid in ["P004", "P008", "P009", "P012"]:
        display(show_retrieval(pid, "llm_screen", k=25))
else:
    print("RUN_LLM=False — skipped")


P004 / llm_screen: missed gold -> ['PP 5.1.1']


,statement_id,gold,status,needs,score,domain,text
0,PP 3.1.1,,eligible,,0.0,3.1 CKD treatment and risk modi fication,Practice Point 3.1.1: Treat people with CKD with a comprehensive treatment strategy to reduce risks of progres
1,PP 3.6.1,✓,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.1: RASi (ACEi or ARB) should be administered using the highest approved dose that is tolera
2,Rec 3.6.3,✓,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Recommendation 3.6.3: We recommend starting RASi (ACEi or ARB) for people with CKD and moderately-to-severely
3,PP 3.6.3,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.3: Hyperkalemia associated with use of RASi can often be managed by measures to reduce the
4,PP 3.6.4,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.4: Continue ACEi or ARB therapy unless serum creatinine rises by more than 30% within 4 wee
5,Rec 3.6.4,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,"Recommendation 3.6.4: We recommend avoiding any combination of ACEi, ARB, and direct renin inhibitor (DRI) the"
6,Rec 3.7.1,✓,eligible,,0.0,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),"Recommendation 3.7.1: We recommend treating patients with type 2 diabetes (T2D), CKD, and an eGFR ≥20 ml/min p"
7,Rec 3.7.2,✓,eligible,,0.0,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),Recommendation 3.7.2: We recommend treating adults with CKD with an SGLT2i for the following (1A):\n eGFR ≥20
8,PP 3.7.1,,eligible,,0.0,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),"Practice Point 3.7.1: Once an SGLT2i is initiated, it is reasonable to continue an SGLT2i even if the eGFR fal"
9,PP 3.8.1,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),Practice Point 3.8.1: Nonsteroidal MRA are most appropriate for adults with T2D who are at high risk of CKD pr


P008 / llm_screen: missed gold -> ['PP 2.1.1', 'PP 3.16.2', 'Rec 2.2.1', 'Rec 3.16.1']


,statement_id,gold,status,needs,score,domain,text
0,PP 4.3.1,✓,eligible,,0.0,4.3 Polypharmacy and drug stewardship,Practice Point 4.3.1: Perform thorough medication review periodically and at transitions of care to assess adh
1,PP 4.1.2,,eligible,,0.0,4.1 Medication choices and monitoring for safety,"Practice Point 4.1.2: Monitor eGFR, electrolytes, and therapeutic medication levels, when indicated, in people"
2,PP 4.2.1,✓,eligible,,0.0,4.2 Dose adjustments by level of GFR,Practice Point 4.2.1: Consider GFR when dosing medications cleared by the kidneys.
3,PP 4.2.2,,eligible,,0.0,4.2 Dose adjustments by level of GFR,"Practice Point 4.2.2: For most people and clinical settings, validated eGFR equations using SCr are appropriat"
4,PP 4.2.3,,eligible,,0.0,4.2 Dose adjustments by level of GFR,"Practice Point 4.2.3: Where more accuracy is required for drug-related decision-making (e.g., dosing due to na"
5,PP 3.6.3,✓,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.3: Hyperkalemia associated with use of RASi can often be managed by measures to reduce the
6,PP 3.8.5,✓,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),"Practice Point 3.8.5: A steroidal MRA may be used for treatment of heart failure, hyperaldosteronism, or refra"
7,PP 3.8.3,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),"Practice Point 3.8.3: To mitigate risk of hyperkalemia, select people with consistently normal serum potassium"
8,PP 3.8.2,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),Practice Point 3.8.2: A nonsteroidal MRA may be added to a RASi and an SGLT2i for treatment of T2D and CKD in
9,Rec 3.7.2,✓,eligible,,0.0,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),Recommendation 3.7.2: We recommend treating adults with CKD with an SGLT2i for the following (1A):\n eGFR ≥20


P009 / llm_screen: missed gold -> ['PP 5.1.1']


,statement_id,gold,status,needs,score,domain,text
0,PP 4.1.4,✓,eligible,,0.0,4.1 Medication choices and monitoring for safety,"Practice Point 4.1.4: When prescribing medications to people with CKD who are of child-bearing potential, alwa"
1,PP 4.1.1,✓,eligible,,0.0,4.1 Medication choices and monitoring for safety,Practice Point 4.1.1: People with CKD may be more susceptible to the nephrotoxic effects of medications. When
2,PP 4.1.2,,eligible,,0.0,4.1 Medication choices and monitoring for safety,"Practice Point 4.1.2: Monitor eGFR, electrolytes, and therapeutic medication levels, when indicated, in people"
3,PP 3.6.5,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.5: Consider reducing the dose or discontinuing ACEi or ARB in the setting of either symptom
4,PP 4.3.3,,eligible,,0.0,4.3 Polypharmacy and drug stewardship,"Practice Point 4.3.3: Consider planned discontinuation of medications (such as metformin, ACEi, ARBs, and SGLT"
5,PP 3.6.1,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.1: RASi (ACEi or ARB) should be administered using the highest approved dose that is tolera
6,Rec 3.6.1,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Recommendation 3.6.1: We recommend starting renin-angiotensin-system inhibitors (RASi) (angiotensin-converting
7,PP 3.6.4,,eligible,,0.0,3.6 Renin-angiotensin system inhibitors,Practice Point 3.6.4: Continue ACEi or ARB therapy unless serum creatinine rises by more than 30% within 4 wee
8,PP 4.3.1,✓,eligible,,0.0,4.3 Polypharmacy and drug stewardship,Practice Point 4.3.1: Perform thorough medication review periodically and at transitions of care to assess adh
9,PP 4.3.1.1,,eligible,,0.0,4.3 Polypharmacy and drug stewardship,Practice Point 4.3.1.1: Educate and inform people with CKD regarding the expected benefits and possible risks


P012 / llm_screen: missed gold -> ['PP 4.2.1']


,statement_id,gold,status,needs,score,domain,text
0,PP 3.15.1.2,,eligible,,0.0,3.15 Cardiovascular disease (CVD) and additional specific interventions to modify risk,"Practice Point 3.15.1.2: In people with CKD, choose statinbased regimens to maximize the absolute reduction in"
1,PP 3.15.1.1,,eligible,,0.0,3.15 Cardiovascular disease (CVD) and additional specific interventions to modify risk,Practice Point 3.15.1.1: Estimate 10-year cardiovascular risk using a validated risk tool.
2,PP 3.7.1,,eligible,,0.0,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),"Practice Point 3.7.1: Once an SGLT2i is initiated, it is reasonable to continue an SGLT2i even if the eGFR fal"
3,PP 3.8.1,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),Practice Point 3.8.1: Nonsteroidal MRA are most appropriate for adults with T2D who are at high risk of CKD pr
4,PP 3.8.2,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),Practice Point 3.8.2: A nonsteroidal MRA may be added to a RASi and an SGLT2i for treatment of T2D and CKD in
5,PP 3.7.3,,eligible,,0.0,3.7 Sodium-glucose cotransporter-2 inhibitors (SGLT2i),Practice Point 3.7.3: SGLT2i initiation or use does not necessitate alteration of frequency of CKD monitoring
6,PP 3.8.3,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),"Practice Point 3.8.3: To mitigate risk of hyperkalemia, select people with consistently normal serum potassium"
7,PP 3.8.4,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),Practice Point 3.8.4: The choice of a nonsteroidal MRA should prioritize agents with documented kidney or card
8,PP 3.8.5,,eligible,,0.0,3.8 Mineralocorticoid receptor antagonists (MRA),"Practice Point 3.8.5: A steroidal MRA may be used for treatment of heart failure, hyperaldosteronism, or refra"
9,Rec 3.15.3.1,,eligible,,0.0,3.15 Cardiovascular disease (CVD) and additional specific interventions to modify risk,"Recommendation 3.15.3.1: We suggest that in stable stress-test confirmed ischemic heart disease, an initial co"


## 13. Evidence package for the generation step

What gets handed to the recommendation LLM: statements grouped by care domain, each with
grade, page, applicability status, **what data is missing**, the section rationale (parent
context) and any linked tables. This replaces `format_guideline_context` in the first notebook.

In [37]:
def build_evidence_package(pid, strategy="filtered_multiquery_rerank", k=15, rationale_chars=1200):
    st = patient_states[pid]
    applicability = filter_statements(st)
    retrieved = retrieve(pid, strategy, k=k)
    by_domain = defaultdict(list)
    for sid, score in retrieved:
        s = stmt_by_id[sid]
        status, _, needs = applicability[sid]
        by_domain[s["domain"]].append({
            "statement_id": sid,
            "grade": s["grade"],
            "page": s["page"],
            "status": status,
            "needs_data": needs,
            "text": s["text"],
            "rationale_excerpt": s["section_rationale"][:rationale_chars],
            "linked_tables": [table_units[t] for t in s["table_refs"] if t in table_units],
        })
    return {
        "patient_state": st.model_dump(),
        "derived": derive(st),
        "evidence_by_domain": dict(by_domain),
    }


def format_evidence_package(pkg):
    lines = [f"PATIENT: {json.dumps(pkg['patient_state'])}", f"DERIVED: {json.dumps(pkg['derived'])}", ""]
    for domain, items_ in pkg["evidence_by_domain"].items():
        lines.append(f"## {domain}")
        for e in items_:
            flag = f" [CONDITIONAL — needs {', '.join(e['needs_data'])}]" if e["needs_data"] else ""
            lines.append(f"- {e['statement_id']} (grade {e['grade'] or 'practice point'}, p.{e['page']}){flag}\n  {e['text']}")
            for t in e["linked_tables"]:
                lines.append(f"  [{t['table_id']}] {t['caption']}\n{t['markdown'][:1500]}")
        lines.append("")
    return "\n".join(lines)


pkg = build_evidence_package("P001")
print(format_evidence_package(pkg)[:4000])

with open(OUTPUT_DIR / "evidence_package_P001.json", "w", encoding="utf-8") as f:
    json.dump(pkg, f, indent=2, ensure_ascii=False, default=str)

PATIENT: {"age": 67, "sex": "M", "egfr": 48.0, "egfr_date": "2026-04-12", "acr_mg_g": null, "potassium": null, "sbp": 158, "known_ckd": false, "hypertension": true, "diabetes": "none", "heart_failure": null, "on_dialysis": false, "kidney_transplant": false, "pregnant": null, "medications": ["Amlodipine 10 mg daily", "amlodipine"], "symptoms": ["fatigue"], "concerns": ["BP remains elevated despite medication", "Fatigue - cause?", "No urine ACR on file", "eGFR lowered (48) - next steps"]}
DERIVED: {"is_adult": true, "gfr_category": "G3a", "acr_category": null, "risk_category": null, "drug_classes": ["CCB"], "missing_data": ["urine ACR"]}

## 1.1 Detection and evaluation of CKD
- PP 1.1.1.2 (grade practice point, p.55)
  Practice Point 1.1.1.2: Following incidental detection of elevated urinary albumin-to-creatinine ratio (ACR), hematuria, or low estimated GFR (eGFR), repeat tests to confirm presence of CKD.
- PP 1.1.3.1 (grade practice point, p.57)
  Practice Point 1.1.3.1: Proof of chro

## Next steps

1. **Clinician review** of `outputs/statement_applicability_review.csv` and the `GOLD` set.
2. Set `RUN_LLM = True` to compare: LLM note extraction, LLM sub-queries, LLM screening (§12).
3. Plug `format_evidence_package` into the `ClinicalFinding` generation cell from `initial_experiment.ipynb`.
4. Move §2–§10 into `src/` modules (`parsing.py`, `applicability.py`, `patient.py`, `retrieval.py`, `evaluation.py`).